In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables across steps for efficiency
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp is integer and create timezone-aware datetime column 'ts' in Asia/Shanghai (UTC+8)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = df['timestamp'].astype(int)
    # interpret epoch as UTC then convert to Asia/Shanghai
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compact overview function
def compact_overview(df, name_field=None):
    total_rows = len(df)
    min_ts = df['ts'].min()
    max_ts = df['ts'].max()
    top_cmdb = df['cmdb_id'].value_counts().head(20).reset_index()
    top_cmdb.columns = ['cmdb_id', 'count']
    if name_field and name_field in df.columns:
        distinct_names = pd.Series(sorted(df[name_field].dropna().unique())[:50], name=name_field)
    else:
        distinct_names = pd.Series([], name=(name_field or 'names'), dtype=object)
    summary = pd.Series({'total_rows': total_rows, 'min_ts': min_ts, 'max_ts': max_ts})
    return summary, top_cmdb, distinct_names

# Generate overviews
metric_summary, metric_top_cmdb, metric_kpi_names = compact_overview(df_metric, 'kpi_name')
trace_summary, trace_top_cmdb, trace_trace_names = compact_overview(df_trace, 'trace_name')
log_summary, log_top_cmdb, log_log_names = compact_overview(df_log, 'log_name')
error_summary, error_top_cmdb, error_names = compact_overview(df_error, None)

# Aggregations: group by (cmdb_id, name) and compute count, p50, p95, p99 for 'value'
def aggregate_metrics(df, name_col):
    # Only keep rows with numeric values in 'value'
    df_ = df.copy()
    df_['value'] = pd.to_numeric(df_['value'], errors='coerce')
    grp = df_.groupby(['cmdb_id', name_col])['value'].agg(
        count='count',
        p50=lambda s: s.quantile(0.5),
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99)
    ).reset_index()
    # round numerical columns for compactness
    grp[['p50','p95','p99']] = grp[['p50','p95','p99']].round(6)
    grp = grp.sort_values('count', ascending=False).reset_index(drop=True)
    return grp

metric_agg = aggregate_metrics(df_metric, 'kpi_name')
trace_agg = aggregate_metrics(df_trace, 'trace_name')
log_agg = aggregate_metrics(df_log, 'log_name')

# Keep top 20 rows for display
metric_agg_top20 = metric_agg.head(20)[['cmdb_id','kpi_name','count','p50','p95','p99']]
trace_agg_top20 = trace_agg.head(20)[['cmdb_id','trace_name','count','p50','p95','p99']]
log_agg_top20 = log_agg.head(20)[['cmdb_id','log_name','count','p50','p95','p99']]

# Time window filtering using Asia/Shanghai timezone (inclusive start, exclusive end)
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))

metric_window_count = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].shape[0]
trace_window_count  = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].shape[0]
log_window_count    = df_log[(df_log['ts'] >= start) & (df_log['ts'] < end)].shape[0]
error_window_count  = df_error[(df_error['ts'] >= start) & (df_error['ts'] < end)].shape[0]

# Final compact outputs (store full intermediate tables in variables above but display compact summaries)
metric_summary, metric_top_cmdb, metric_kpi_names, metric_agg_top20, metric_window_count, \
trace_summary, trace_top_cmdb, trace_trace_names, trace_agg_top20, trace_window_count, \
log_summary, log_top_cmdb, log_log_names, log_agg_top20, log_window_count, \
error_summary, error_top_cmdb, error_window_count

```
Out[1]:
```
```python
# Summarize execution results in plain English using existing kernel variables.
# The final displayed output is the single string `summary`.

# Reuse variables from previous run: metric_summary, metric_top_cmdb, metric_kpi_names,
# metric_agg_top20, metric_window_count, trace_summary, trace_top_cmdb, trace_trace_names,
# trace_agg_top20, trace_window_count, log_summary, log_top_cmdb, log_log_names,
# log_agg_top20, log_window_count, error_summary, error_top_cmdb, error_window_count

def top_list_str(df_top, n=5):
    # format top n cmdb_id(count)
    rows = df_top.head(n)
    return ", ".join(f"{r.cmdb_id}({int(r.count)})" for r in rows.itertuples())

summary = (
    "Summary of telemetry files (timestamps in UTC+8):\n\n"
    "metric.csv:\n"
    f"- total rows: {int(metric_summary['total_rows'])}\n"
    f"- time range: {metric_summary['min_ts']} to {metric_summary['max_ts']}\n"
    f"- top cmdb_ids (top 5): {top_list_str(metric_top_cmdb,5)}\n"
    f"- distinct kpi_name samples (up to 50 shown): {len(metric_kpi_names)} names listed\n"
    f"- aggregation (by cmdb_id,kpi_name) computed; top aggregated rows shown previously (top count ~{int(metric_agg_top20.iloc[0].count) if not metric_agg_top20.empty else 'N/A'})\n"
    f"- rows in window 2022-03-21 07:30:00 to 08:00:00 (inclusive start, exclusive end): {int(metric_window_count)}\n\n"
    "trace.csv:\n"
    f"- total rows: {int(trace_summary['total_rows'])}\n"
    f"- time range: {trace_summary['min_ts']} to {trace_summary['max_ts']}\n"
    f"- top cmdb_ids (top 5): {top_list_str(trace_top_cmdb,5)}\n"
    f"- distinct trace_name samples (up to 50 shown): {len(trace_trace_names)} names listed\n"
    f"- aggregation (by cmdb_id,trace_name) computed; top aggregated rows shown previously (top count ~{int(trace_agg_top20.iloc[0].count) if not trace_agg_top20.empty else 'N/A'})\n"
    f"- rows in window 2022-03-21 07:30:00 to 08:00:00: {int(trace_window_count)}\n\n"
    "log.csv:\n"
    f"- total rows: {int(log_summary['total_rows'])}\n"
    f"- time range: {log_summary['min_ts']} to {log_summary['max_ts']}\n"
    f"- top cmdb_ids (top 5): {top_list_str(log_top_cmdb,5)}\n"
    f"- distinct log_name values: {', '.join(list(log_log_names.astype(str)))}\n"
    f"- aggregation (by cmdb_id,log_name) computed; top aggregated rows shown previously (top count ~{int(log_agg_top20.iloc[0].count) if not log_agg_top20.empty else 'N/A'})\n"
    f"- rows in window 2022-03-21 07:30:00 to 08:00:00: {int(log_window_count)}\n\n"
    "error_logs.csv:\n"
    f"- total rows: {int(error_summary['total_rows'])}\n"
    f"- time range: {error_summary['min_ts']} to {error_summary['max_ts']}\n"
    f"- top cmdb_ids: (none)\n"
    f"- rows in window 2022-03-21 07:30:00 to 08:00:00: {int(error_window_count)}\n\n"
    "Notes:\n"
    "- All aggregations (count, p50, p95, p99) were computed over the full series (no time filtering) as requested.\n"
    "- The displayed top lists and counts are compact summaries; full intermediate tables are retained in kernel variables."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(total_rows                       322448
min_ts        2022-03-21 07:00:00+08:00
max_ts        2022-03-21 08:00:00+08:00
dtype: object,                      cmdb_id  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  12200
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6         checkoutservice2-0   9028
7          checkoutservice-1   8784
8          checkoutservice-2   8540
9          checkoutservice-0   8540
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14              adservice2-0   6222
15  recommendationservice2-0   6222
16            cartservice2-0   6100
17             cartservice-2   6100
18             cartservice-0   6100
19   recommendationservice-2   5978, 0                                        app.grpc.count
1                                          app.grpc.mrt
2                                           app.grpc.rr
3                                           app.grpc.sr
4                                        app.http.count
                            ...                        
45               container.node-5.container_memory_swap
46           container.node-5.container_memory_usage_MB
47     container.node-5.container_memory_working_set_MB
48    container.node-5.container_network_receive_MB....
49    container.node-5.container_network_receive_err...
Name: kpi_name, Length: 50, dtype: object,                cmdb_id                                           kpi_name  count           p50           p95           p99
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61      0.000000      0.000000      0.000000
1            adservice                                     app.grpc.count     61     76.000000     93.000000     99.200000
2            adservice                                       app.grpc.mrt     61      0.002579      0.002796      0.002852
3            adservice                                        app.grpc.rr     61    100.000000    100.000000    100.000000
4            adservice                                        app.grpc.sr     61    100.000000    100.000000    100.000000
5            adservice                                     app.http.count     61     24.000000     24.000000     24.000000
6            adservice                                       app.http.mrt     61      0.009840      0.011081      0.127453
7            adservice                                        app.http.rr     61    100.000000    100.000000    100.000000
8            adservice                                        app.http.sr     61    100.000000    100.000000    100.000000
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount     61   5162.000000   5162.000000   5162.000000
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     61   5185.000000   5185.000000   5185.000000
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     61     23.000000     23.000000     23.000000
12           adservice             runtime.java_lang_ClassLoading_Verbose     61      0.000000      0.000000      0.000000
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...     61      1.000000      1.000000      1.000000
14           adservice  runtime.java_lang_Compilation_TotalCompilation...     61  47021.000000  47021.000000  47021.000000
15           adservice  runtime.java_lang_GarbageCollector_CollectionC...     61  38394.666667  38467.083333  38473.483333
16           adservice  runtime.java_lang_GarbageCollector_CollectionC...     61      8.000000      8.000000      8.000000
17           adservice  runtime.java_lang_GarbageCollector_CollectionT...     61  69699.916667  69833.166667  69845.266667
18  shippingservice2-0  container.node-6.container_network_transmit_MB...     61      0.538112      0.544687      0.567594
19  shippingservice2-0  container.node-6.container_network_transmit_er...     61      0.000000      0.000000      0.000000, 158580, total_rows                        27912
min_ts        2022-03-21 07:30:00+08:00
max_ts        2022-03-21 07:59:00+08:00
dtype: object,                     cmdb_id  count
0                frontend-1   2624
1                frontend-0   2620
2                frontend-2   2484
3         checkoutservice-0   2084
4         checkoutservice-2   2012
5         checkoutservice-1   1984
6   productcatalogservice-1   1016
7   productcatalogservice-0   1008
8   productcatalogservice-2   1004
9   recommendationservice-1    840
10  recommendationservice-0    840
11  recommendationservice-2    840
12            cartservice-0    792
13            cartservice-2    772
14            cartservice-1    760
15        currencyservice-1    696
16        currencyservice-0    684
17        currencyservice-2    684
18        shippingservice-1    644
19        shippingservice-2    628, 0     trace.from_checkoutservice-0.duration_mean
1      trace.from_checkoutservice-0.duration_p95
2        trace.from_checkoutservice-0.error_rate
3         trace.from_checkoutservice-0.row_count
4     trace.from_checkoutservice-1.duration_mean
                         ...                    
45             trace.to_adservice-0.duration_p95
46               trace.to_adservice-0.error_rate
47                trace.to_adservice-0.row_count
48            trace.to_adservice-1.duration_mean
49             trace.to_adservice-1.duration_p95
Name: trace_name, Length: 50, dtype: object,                     cmdb_id                                      trace_name  count        p50        p95        p99
0               adservice-1                 trace.from_frontend-1.row_count     30   9.000000  14.000000  14.000000
1                frontend-0  trace.to_productcatalogservice-2.duration_mean     30   0.004840   0.005035   0.005099
2                frontend-0   trace.to_productcatalogservice-2.duration_p95     30   0.006103   0.006666   0.006700
3                frontend-0     trace.to_productcatalogservice-2.error_rate     30   0.000000   0.000000   0.000000
4                frontend-0      trace.to_productcatalogservice-2.row_count     30  78.500000  90.550000  91.710000
5                frontend-0  trace.to_recommendationservice-1.duration_mean     30   0.002777   0.033414   0.057391
6                frontend-0   trace.to_recommendationservice-1.duration_p95     30   0.003198   0.108719   0.230813
7                frontend-0     trace.to_recommendationservice-1.error_rate     30   0.000000   0.000000   0.000000
8                frontend-0      trace.to_recommendationservice-1.row_count     30  12.000000  14.000000  14.000000
9                frontend-0  trace.to_productcatalogservice-1.duration_mean     30   0.004844   0.005322   0.005382
10               frontend-0   trace.to_productcatalogservice-1.duration_p95     30   0.005992   0.006792   0.007326
11               frontend-0     trace.to_productcatalogservice-1.error_rate     30   0.000000   0.000000   0.000000
12               frontend-0      trace.to_productcatalogservice-1.row_count     30  78.500000  90.550000  91.000000
13               frontend-0  trace.to_recommendationservice-0.duration_mean     30   0.002640   0.064046   0.089144
14               frontend-0   trace.to_recommendationservice-0.duration_p95     30   0.003084   0.215476   0.294054
15               frontend-0     trace.to_recommendationservice-0.error_rate     30   0.000000   0.000000   0.000000
16               frontend-0      trace.to_recommendationservice-0.row_count     30  12.000000  14.000000  14.000000
17  recommendationservice-2  trace.to_productcatalogservice-0.duration_mean     30   0.000030   0.006167   0.012135
18  recommendationservice-2   trace.to_productcatalogservice-0.duration_p95     30   0.000035   0.033146   0.063627
19  recommendationservice-2     trace.to_productcatalogservice-0.error_rate     30   0.000000   0.000000   0.000000, 27912, total_rows                         1794
min_ts        2022-03-21 07:30:00+08:00
max_ts        2022-03-21 08:00:00+08:00
dtype: object,                     cmdb_id  count
0               adservice-1     62
1                frontend-1     62
2                frontend-0     62
3   productcatalogservice-2     62
4             cartservice-0     60
5               adservice-0     60
6               adservice-2     60
7         checkoutservice-2     60
8         currencyservice-0     60
9         currencyservice-1     60
10        currencyservice-2     60
11               frontend-2     60
12            cartservice-1     60
13            cartservice-2     60
14        checkoutservice-0     60
15        shippingservice-0     60
16        shippingservice-1     60
17        shippingservice-2     60
18  recommendationservice-2     60
19  productcatalogservice-0     60, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object,                     cmdb_id         log_name  count    p50     p95     p99
0               adservice-1    log.row_count     31   50.0   63.00   64.00
1               adservice-1  log.error_count     31    0.0    0.00    0.00
2                frontend-1    log.row_count     31  774.0  903.00  940.20
3                frontend-1  log.error_count     31    0.0    0.00    0.00
4                frontend-0    log.row_count     31  786.0  899.50  933.10
5                frontend-0  log.error_count     31    0.0    0.00    0.00
6   productcatalogservice-2  log.error_count     31    0.0    0.00    0.00
7   productcatalogservice-2    log.row_count     31  477.0  502.50  525.60
8             cartservice-0  log.error_count     30    0.0    0.00    0.00
9             cartservice-0    log.row_count     30  406.0  450.55  456.68
10              adservice-0    log.row_count     30   64.0   76.00   78.84
11              adservice-0  log.error_count     30    0.0    0.00    0.00
12              adservice-2  log.error_count     30    0.0    0.00    0.00
13              adservice-2    log.row_count     30   51.0   64.00   65.42
14        checkoutservice-2    log.row_count     30   34.0   48.00   49.42
15        checkoutservice-2  log.error_count     30    0.0    0.00    0.00
16        currencyservice-0    log.row_count     30  353.0  403.75  429.43
17        currencyservice-0  log.error_count     30    0.0    0.00    0.00
18        currencyservice-1  log.error_count     30    0.0    0.00    0.00
19        currencyservice-1    log.row_count     30  350.5  414.35  435.49, 1786, total_rows      0
min_ts        NaT
max_ts        NaT
dtype: object, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0)```
```

In [ ]:
# In[2]:

# Re-run with corrected timestamp handling for display.
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Prepare df_metric
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Include only candidate cmdb_ids (exclude mesh/system-like components)
exclude_pattern = r"mesh|jaeger|collector|kube|istio|prometheus|sidecar|envoy|gateway"
mask_include = ~df_metric['cmdb_id'].astype(str).str.contains(exclude_pattern, case=False, na=False)
df_metric_clean = df_metric[mask_include].copy()

# Compute thresholds (global P95 and P5) per (cmdb_id, kpi_name) over full series
thresholds = (
    df_metric_clean
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
)

# 2) Filter to incident window (inclusive start, exclusive end) in UTC+8
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))
df_window = df_metric_clean[(df_metric_clean['ts'] >= start) & (df_metric_clean['ts'] < end)].copy()

# 3) Merge thresholds into window rows and compute per-group stats in window
df_win_merge = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

def compute_group_stats(g):
    P95 = g['P95'].iloc[0]
    P5  = g['P5'].iloc[0]
    vals = g['value']
    ts = g['ts']
    # highs
    is_high = vals > P95
    count_high = int(is_high.sum())
    earliest_high_ts = ts[is_high].min() if count_high>0 else pd.NaT
    max_value_in_window = vals.max() if not vals.empty else np.nan
    severity_high = (max_value_in_window - P95) / (P95 + 1e-12) if not np.isnan(max_value_in_window) else np.nan
    # lows
    is_low = vals < P5
    count_low = int(is_low.sum())
    earliest_low_ts = ts[is_low].min() if count_low>0 else pd.NaT
    min_value_in_window = vals.min() if not vals.empty else np.nan
    severity_low = (P5 - min_value_in_window) / (abs(P5) + 1e-12) if not np.isnan(min_value_in_window) else np.nan
    return pd.Series({
        'count_high': count_high,
        'earliest_high_ts': earliest_high_ts,
        'max_value_in_window': max_value_in_window,
        'severity_high': severity_high,
        'count_low': count_low,
        'earliest_low_ts': earliest_low_ts,
        'min_value_in_window': min_value_in_window,
        'severity_low': severity_low
    })

if not df_win_merge.empty:
    window_stats = df_win_merge.groupby(['cmdb_id','kpi_name']).apply(compute_group_stats).reset_index()
else:
    window_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','count_high','earliest_high_ts',
                                         'max_value_in_window','severity_high','count_low',
                                         'earliest_low_ts','min_value_in_window','severity_low'])

# 3b) Combine with thresholds to include series with no data in window
metric_anomaly_full = thresholds.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

# Normalize/clean columns
metric_anomaly_full['count_high'] = metric_anomaly_full['count_high'].fillna(0).astype(int)
metric_anomaly_full['count_low']  = metric_anomaly_full['count_low'].fillna(0).astype(int)
# Keep earliest timestamps as timezone-aware or NaT in metric_anomaly_full (for later use)
# Round numeric fields for compactness
metric_anomaly_full['severity_high'] = metric_anomaly_full['severity_high'].astype(float).round(6)
metric_anomaly_full['severity_low']  = metric_anomaly_full['severity_low'].astype(float).round(6)
metric_anomaly_full['P95'] = metric_anomaly_full['P95'].round(6)
metric_anomaly_full['P5']  = metric_anomaly_full['P5'].round(6)

# 4) Top 20 by max(count_high,count_low)
metric_anomaly_full['max_count'] = metric_anomaly_full[['count_high','count_low']].max(axis=1)
metric_anomaly_full_sorted = metric_anomaly_full.sort_values(['max_count','count_high','count_low'], ascending=False)

anomaly_top20 = metric_anomaly_full_sorted.head(20)[
    ['cmdb_id','kpi_name','count_high','earliest_high_ts','severity_high',
     'count_low','earliest_low_ts','severity_low','P95','P5']
].copy()

# Format earliest timestamps as strings in UTC+8 for compact display (keep originals in metric_anomaly_full)
def fmt_ts(x):
    if pd.isna(x):
        return ""
    # ensure tz-aware then convert to Asia/Shanghai and format
    if getattr(x, "tzinfo", None) is None:
        # assume already in tz-aware via earlier processing; if naive, attach tz
        x = tz.localize(pd.Timestamp(x))
    try:
        x_local = x.tz_convert(tz)
    except Exception:
        x_local = x
    return x_local.strftime("%Y-%m-%d %H:%M:%S %z")

anomaly_top20['earliest_high_ts'] = anomaly_top20['earliest_high_ts'].apply(fmt_ts)
anomaly_top20['earliest_low_ts']  = anomaly_top20['earliest_low_ts'].apply(fmt_ts)

# Summary counts
total_series_scanned = int(thresholds.shape[0])
series_with_anomaly = int(metric_anomaly_full[(metric_anomaly_full['count_high']>0) | (metric_anomaly_full['count_low']>0)].shape[0])

# Display compact outputs (top 20 rows)
anomaly_top20, total_series_scanned, series_with_anomaly

```
Out[2]:
```
```python
# Summarize results in plain English (UTC+8). Display as the variable `summary`.
summary = (
    "Metric analysis summary (UTC+8):\n\n"
    "What I did:\n"
    "- Computed global P95 and P5 thresholds for each (cmdb_id, kpi_name) over the entire metric series (no time filtering).\n"
    "- Filtered metric data to the incident window 2022-03-21 07:30:00 (inclusive) to 2022-03-21 08:00:00 (exclusive) UTC+8.\n"
    "- For each series, counted high-side breaches (>P95) and low-side breaches (<P5), recorded earliest breach timestamps, and computed severity scores.\n\n"
    "High-level results:\n"
    f"- Total distinct (cmdb_id, kpi_name) series scanned: 5298\n"
    f"- Series with at least one anomaly in the window (count_high>0 or count_low>0): 1393\n\n"
    "Top example anomalous series (top by number of breaches):\n"
    "1) checkoutservice-1 | container.node-6.container_network_receive_MB...  -> "
    "count_high=3 (earliest 2022-03-21 07:32:00 +0800), severity_high=10.087109; "
    "count_low=3 (earliest 2022-03-21 07:40:00 +0800), severity_low=0.128642; P95=0.056048, P5=0.029524\n"
    "2) currencyservice-1 | mesh.destination.checkoutservice.currencyservice...  -> "
    "count_high=3 (earliest 2022-03-21 07:40:00 +0800), severity_high=0.202696; "
    "count_low=3 (earliest 2022-03-21 07:45:00 +0800), severity_low=0.398134; P95=4822.5, P5=1607.5\n"
    "3) currencyservice-2 | container.node-6.container_cpu_user_seconds  -> "
    "count_high=3 (earliest 2022-03-21 07:32:00 +0800), severity_high=0.133333; "
    "count_low=3 (earliest 2022-03-21 07:33:00 +0800), severity_low=0.052632; P95=0.15, P5=0.095\n"
    "4) frontend-1 | mesh.source.frontend.adservice.istio_request_b...  -> "
    "count_high=3 (earliest 2022-03-21 07:35:00 +0800), severity_high=0.091140; "
    "count_low=3 (earliest 2022-03-21 07:36:00 +0800), severity_low=0.123626; P95=58975.0, P5=36400.0\n"
    "5) node-6 | system.cpu.iowait  -> "
    "count_high=3 (earliest 2022-03-21 07:49:00 +0800), severity_high=0.050428; "
    "count_low=3 (earliest 2022-03-21 07:54:00 +0800), severity_low=1.0; P95=21.02, P5=0.04\n\n"
    "Notes:\n"
    "- Timestamps shown are in Asia/Shanghai (UTC+8).\n"
    "- The full intermediate table with all per-series thresholds and window stats is retained in kernel variable `metric_anomaly_full`.\n"
    "- The compact top-20 result is available in kernel variable `anomaly_top20`.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                           kpi_name  count_high           earliest_high_ts  severity_high  count_low            earliest_low_ts  severity_low            P95             P5
1634        checkoutservice-1  container.node-6.container_network_receive_MB....           3  2022-03-21 07:32:00 +0800      10.087109          3  2022-03-21 07:40:00 +0800      0.128642       0.056048       0.029524
2174        currencyservice-1  mesh.destination.checkoutservice.currencyservi...           3  2022-03-21 07:40:00 +0800       0.202696          3  2022-03-21 07:45:00 +0800      0.398134    4822.500000    1607.500000
2196        currencyservice-2        container.node-6.container_cpu_user_seconds           3  2022-03-21 07:32:00 +0800       0.133333          3  2022-03-21 07:33:00 +0800      0.052632       0.150000       0.095000
2947               frontend-1  mesh.source.frontend.adservice.istio_request_b...           3  2022-03-21 07:35:00 +0800       0.091140          3  2022-03-21 07:36:00 +0800      0.123626   58975.000000   36400.000000
2957               frontend-1  mesh.source.frontend.adservice.istio_response_...           3  2022-03-21 07:35:00 +0800       0.086453          3  2022-03-21 07:36:00 +0800      0.068354   12955.000000    7900.000000
2978               frontend-1  mesh.source.frontend.checkoutservice.istio_req...           3  2022-03-21 07:56:00 +0800       3.384877          3  2022-03-21 07:45:00 +0800      0.621212       0.294250       0.049500
3545                   node-2                       system.net.udp.out_datagrams           3  2022-03-21 07:33:00 +0800       0.102804          3  2022-03-21 07:39:00 +0800      0.021053       1.070000       0.950000
3736                   node-6                                  system.cpu.iowait           3  2022-03-21 07:49:00 +0800       0.050428          3  2022-03-21 07:54:00 +0800      1.000000      21.020000       0.040000
3750                   node-6                                    system.io.await           3  2022-03-21 07:46:00 +0800       0.227879          3  2022-03-21 07:54:00 +0800      1.000000      24.750000       0.520000
3754                   node-6                                    system.io.svctm           3  2022-03-21 07:46:00 +0800       0.263158          3  2022-03-21 07:54:00 +0800      1.000000       0.950000       0.100000
4168  productcatalogservice-0  mesh.destination.frontend.productcatalogservic...           3  2022-03-21 07:57:00 +0800       2.934935          3  2022-03-21 07:42:00 +0800      0.047653      25.414850       0.900250
4274  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...           3  2022-03-21 07:42:00 +0800       0.086162          3  2022-03-21 07:31:00 +0800      0.721763       0.028725       0.009075
4814  recommendationservice-2  mesh.destination.frontend.recommendationservic...           3  2022-03-21 07:57:00 +0800       0.887036          3  2022-03-21 07:42:00 +0800      0.046209       0.619000       0.097925
4830  recommendationservice-2  mesh.source.recommendationservice.productcatal...           3  2022-03-21 07:57:00 +0800       0.996506          3  2022-03-21 07:44:00 +0800      0.033689       0.551000       0.030425
118                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...           3  2022-03-21 07:32:00 +0800       0.165368          2  2022-03-21 07:41:00 +0800      0.124641  409122.666667  132631.333333
246                 adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...           3  2022-03-21 07:51:00 +0800       0.014986          2  2022-03-21 07:58:00 +0800      0.675636       4.504167       1.145833
503               adservice-1  mesh.destination.frontend.adservice.istio_requ...           3  2022-03-21 07:30:00 +0800       0.011962          2  2022-03-21 07:58:00 +0800      0.037676       0.083600       0.055075
853                adservice2  runtime.java_lang_OperatingSystem_SystemLoadAv...           3  2022-03-21 07:51:00 +0800       0.023126          2  2022-03-21 07:58:00 +0800      0.675636       4.468333       1.145833
1348            cartservice-2  mesh.source.cartservice.redis-cart.istio_tcp_r...           3  2022-03-21 07:30:00 +0800       0.018868          2  2022-03-21 07:58:00 +0800      0.152666    3922.000000    2803.500000
1806        checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...           3  2022-03-21 07:57:00 +0800       1.241322          2  2022-03-21 07:43:00 +0800      1.000000       0.302500       0.023750, 5298, 1393)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse timezone and dataframes if available
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric exists
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp, value, ts columns
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure thresholds variable exists (P95,P5 per series). If not, compute (global over full series).
try:
    thresholds
except NameError:
    # Exclude mesh/system-like cmdb_ids as before
    exclude_pattern = r"mesh|jaeger|collector|kube|istio|prometheus|sidecar|envoy|gateway"
    mask_include = ~df_metric['cmdb_id'].astype(str).str.contains(exclude_pattern, case=False, na=False)
    df_metric_clean = df_metric[mask_include].copy()
    thresholds = (
        df_metric_clean
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    )

# Prepare window bounds (UTC+8)
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))

# Build df_window from df_metric but include only candidate cmdb_ids consistent with thresholds
# (join df_metric with thresholds to restrict to series that have thresholds)
df_metric_thresh_join = df_metric.merge(thresholds[['cmdb_id','kpi_name']], on=['cmdb_id','kpi_name'], how='inner')

df_window = df_metric_thresh_join[(df_metric_thresh_join['ts'] >= start) & (df_metric_thresh_join['ts'] < end)].copy()

# Merge thresholds (P95,P5) into window rows
df_win_merge = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine anomalous rows
df_win_merge['is_high'] = df_win_merge['value'] > df_win_merge['P95']
df_win_merge['is_low']  = df_win_merge['value'] < df_win_merge['P5']
df_anom = df_win_merge[df_win_merge['is_high'] | df_win_merge['is_low']].copy()

# Assign fault_type
df_anom['fault_type'] = np.where(df_anom['is_high'], 'high', 'low')

# Sort for grouping
df_anom = df_anom.sort_values(['cmdb_id','kpi_name','fault_type','ts']).reset_index(drop=True)

# If no anomalies, create empty metric_faults_full
if df_anom.empty:
    metric_faults_full = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_type','start_ts','end_ts','duration_minutes',
        'num_points','peak_deviation','peak_value','threshold_used'
    ])
else:
    # For each (cmdb_id,kpi_name,fault_type) detect consecutive-minute segments
    def assign_segment_ids(g):
        # g is sorted by ts
        g = g.sort_values('ts').copy()
        # compute difference in minutes
        dt = g['ts'].diff().fillna(pd.Timedelta(seconds=0))
        # new segment when diff != 1 minute
        new_seg = (dt != pd.Timedelta(minutes=1))
        seg_id = new_seg.cumsum()
        g['seg_id'] = seg_id
        return g

    df_segments = df_anom.groupby(['cmdb_id','kpi_name','fault_type'], group_keys=False).apply(assign_segment_ids).reset_index(drop=True)

    # Aggregate per segment
    agg_rows = []
    gb = df_segments.groupby(['cmdb_id','kpi_name','fault_type','seg_id'], as_index=False)
    for name, group in gb:
        cmdb_id, kpi_name, fault_type, seg = name
        start_ts = group['ts'].min()
        end_ts = group['ts'].max()
        duration_minutes = int((end_ts - start_ts) / pd.Timedelta(minutes=1)) + 1
        num_points = int(group.shape[0])
        if fault_type == 'high':
            # peak_deviation = max((value - P95)/(P95+1e-12))
            deviations = (group['value'] - group['P95']) / (group['P95'] + 1e-12)
            peak_deviation = float(deviations.max())
            peak_value = float(group['value'].max())
            threshold_used = float(group['P95'].iloc[0])
        else:
            deviations = (group['P5'] - group['value']) / (group['P5'].abs() + 1e-12)
            peak_deviation = float(deviations.max())
            peak_value = float(group['value'].min())
            threshold_used = float(group['P5'].iloc[0])
        agg_rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'fault_type': fault_type,
            'start_ts': start_ts,
            'end_ts': end_ts,
            'duration_minutes': duration_minutes,
            'num_points': num_points,
            'peak_deviation': peak_deviation,
            'peak_value': peak_value,
            'threshold_used': threshold_used
        })

    metric_faults_full = pd.DataFrame(agg_rows)

# Round numeric fields for compactness
if not metric_faults_full.empty:
    metric_faults_full['peak_deviation'] = metric_faults_full['peak_deviation'].round(6)
    metric_faults_full['peak_value'] = metric_faults_full['peak_value'].round(6)
    metric_faults_full['threshold_used'] = metric_faults_full['threshold_used'].round(6)

# Format timestamps for display (keep originals in metric_faults_full)
# Create display columns
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

if not metric_faults_full.empty:
    metric_faults_full['start_ts_str'] = metric_faults_full['start_ts'].apply(fmt_ts)
    metric_faults_full['end_ts_str'] = metric_faults_full['end_ts'].apply(fmt_ts)

# (A) Top 20 faults sorted by peak_deviation descending
if not metric_faults_full.empty:
    faults_top20 = metric_faults_full.sort_values('peak_deviation', ascending=False).head(20)[
        ['cmdb_id','kpi_name','fault_type','start_ts_str','end_ts_str','duration_minutes',
         'num_points','peak_deviation','peak_value','threshold_used']
    ].rename(columns={'start_ts_str':'start_ts','end_ts_str':'end_ts'})
else:
    faults_top20 = metric_faults_full.head(0)[[
        'cmdb_id','kpi_name','fault_type','start_ts','end_ts','duration_minutes',
        'num_points','peak_deviation','peak_value','threshold_used'
    ]]

# (B) Per-component summary: cmdb_id, total_faults, earliest_fault_start (UTC+8)
if not metric_faults_full.empty:
    comp_summary = metric_faults_full.groupby('cmdb_id', as_index=False).agg(
        total_faults=('cmdb_id','count'),
        earliest_fault_start_ts=('start_ts','min')
    )
    comp_summary['earliest_fault_start'] = comp_summary['earliest_fault_start_ts'].apply(fmt_ts)
    comp_summary_top20 = comp_summary.sort_values('total_faults', ascending=False).head(20)[
        ['cmdb_id','total_faults','earliest_fault_start']
    ]
else:
    comp_summary_top20 = pd.DataFrame(columns=['cmdb_id','total_faults','earliest_fault_start'])

# Keep full table in kernel variable metric_faults_full (already set)
# Return compact outputs (two variables)
faults_top20, comp_summary_top20

```
Out[3]:
```
```python
# Plain-English summary of the fault-segmentation results (UTC+8).
summary = (
    "Fault segmentation summary (UTC+8):\n\n"
    "What was produced:\n"
    "- All anomalous rows (value > P95 => 'high', value < P5 => 'low') inside 2022-03-21 07:30:00 to 08:00:00 were grouped into consecutive-minute fault segments per (cmdb_id, kpi_name, fault_type).\n"
    "- The full faults table is stored in kernel variable `metric_faults_full`.\n"
    "- The top-20 faults by peak_deviation are available in `faults_top20` and a per-component summary (top 20 components) is in `comp_summary_top20`.\n\n"
    "Key findings (from returned outputs):\n"
    "- Many detected faults are single-minute segments (duration 1 minute); a few span 2 minutes in the top list.\n"
    "- The highest peak_deviation entries are for series on adservice2-0 and productcatalogservice-2 and similar; examples:\n"
    "  * adservice2-0 | mesh.source.adservice2.basic-tidb... (high)  start=2022-03-21 07:30:00 +0800, peak_deviation ≈ 1.229e+15, peak_value=1229, threshold_used=0.0\n"
    "  * Several other top faults also show extremely large peak_deviation because their threshold_used (P95 or P5) is 0.0, which inflates the deviation ratio. Interpret those with caution.\n"
    "- Example of non-zero-threshold faults: node-5 | system.io.rkb_s (high) at 2022-03-21 07:38:00, peak_value=48, threshold_used=0.0 (also shows zero threshold) — again indicates many thresholds are zero for certain KPIs.\n\n"
    "Per-component overview (top components by fault count, from returned table):\n"
    "- frontend-0: 214 faults (earliest 2022-03-21 07:30:00 +0800)\n"
    "- frontend-1: 181 faults (earliest 2022-03-21 07:31:00 +0800)\n"
    "- checkoutservice-1: 140 faults (earliest 2022-03-21 07:30:00 +0800)\n"
    "- currencyservice-0: 110 faults (earliest 2022-03-21 07:30:00 +0800)\n"
    "- frontend-2, productcatalogservice-1, productcatalogservice-0, adservice2, ... appear next in the top-20 list (see `comp_summary_top20`).\n\n"
    "Actionable notes:\n"
    "- Many series have P95 or P5 equal to 0.0, producing extremely large deviation ratios; inspect these series' raw values and threshold computation (zero thresholds may be expected for counters or sparse metrics).\n"
    "- Focus initial investigation on components with the largest number of fault segments (frontends and checkoutservice nodes) and on faults with large peak_deviation but non-zero thresholds.\n\n"
    "Data locations in the kernel:\n"
    "- Full faults table: metric_faults_full\n"
    "- Top-20 faults (displayed): faults_top20\n"
    "- Per-component summary (top 20): comp_summary_top20\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                           kpi_name fault_type                   start_ts                     end_ts  duration_minutes  num_points  peak_deviation   peak_value  threshold_used
358              adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_se...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    1.229000e+15  1229.000000             0.0
357              adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_re...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    6.540000e+14   654.000000             0.0
2605  productcatalogservice-2  container.node-6.container_memory_failures.hie...       high  2022-03-21 07:49:00 +0800  2022-03-21 07:49:00 +0800                 1           1    9.900000e+13    99.000000             0.0
2603  productcatalogservice-2  container.node-6.container_memory_failures.con...       high  2022-03-21 07:49:00 +0800  2022-03-21 07:49:00 +0800                 1           1    9.900000e+13    99.000000             0.0
2123                   node-5                                    system.io.rkb_s       high  2022-03-21 07:38:00 +0800  2022-03-21 07:38:00 +0800                 1           1    4.800000e+13    48.000000             0.0
2604  productcatalogservice-2  container.node-6.container_memory_failures.hie...       high  2022-03-21 07:45:00 +0800  2022-03-21 07:45:00 +0800                 1           1    3.300000e+13    33.000000             0.0
2602  productcatalogservice-2  container.node-6.container_memory_failures.con...       high  2022-03-21 07:45:00 +0800  2022-03-21 07:45:00 +0800                 1           1    3.300000e+13    33.000000             0.0
328              adservice2-0     container.node-6.container_fs_inodes./dev/vda1       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    2.600000e+13    26.000000             0.0
329              adservice2-0      container.node-6.container_fs_writes./dev/vda       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    2.500000e+13    25.000000             0.0
2488  productcatalogservice-1  container.node-6.container_memory_failures.hie...       high  2022-03-21 07:37:00 +0800  2022-03-21 07:38:00 +0800                 2           2    1.650000e+13    16.500000             0.0
2487  productcatalogservice-1  container.node-6.container_memory_failures.con...       high  2022-03-21 07:37:00 +0800  2022-03-21 07:38:00 +0800                 2           2    1.650000e+13    16.500000             0.0
2385  productcatalogservice-0      container.node-6.container_fs_writes./dev/vda       high  2022-03-21 07:55:00 +0800  2022-03-21 07:55:00 +0800                 1           1    8.000000e+12     8.000000             0.0
2122                   node-5                                      system.io.r_s       high  2022-03-21 07:38:00 +0800  2022-03-21 07:38:00 +0800                 1           1    2.500000e+12     2.500000             0.0
364             cartservice-0  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-21 07:39:00 +0800  2022-03-21 07:39:00 +0800                 1           1    2.000000e+12     2.000000             0.0
399             cartservice-1  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-21 07:59:00 +0800  2022-03-21 07:59:00 +0800                 1           1    2.000000e+12     2.000000             0.0
462             cartservice-2  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-21 07:41:00 +0800  2022-03-21 07:41:00 +0800                 1           1    1.000000e+12     1.000000             0.0
994         currencyservice-1  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-21 07:38:00 +0800  2022-03-21 07:38:00 +0800                 1           1    1.000000e+12     1.000000             0.0
289                adservice2                       runtime.jvm_classes_unloaded       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    6.666667e+11     0.666667             0.0
356              adservice2-0  mesh.source.adservice2.basic-tidb.istio_tcp_co...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800                 1           1    5.000000e+11     0.500000             0.0
1086        currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-21 07:48:00 +0800  2022-03-21 07:49:00 +0800                 2           2    5.000000e+11     0.500000             0.0,                     cmdb_id  total_faults       earliest_fault_start
27               frontend-0           214  2022-03-21 07:30:00 +0800
28               frontend-1           181  2022-03-21 07:31:00 +0800
13        checkoutservice-1           140  2022-03-21 07:30:00 +0800
17        currencyservice-0           110  2022-03-21 07:30:00 +0800
29               frontend-2           109  2022-03-21 07:30:00 +0800
44  productcatalogservice-1           109  2022-03-21 07:30:00 +0800
43  productcatalogservice-0           102  2022-03-21 07:30:00 +0800
4                adservice2           100  2022-03-21 07:30:00 +0800
45  productcatalogservice-2            99  2022-03-21 07:31:00 +0800
19        currencyservice-2            96  2022-03-21 07:31:00 +0800
18        currencyservice-1            91  2022-03-21 07:33:00 +0800
31                   node-1            86  2022-03-21 07:30:00 +0800
14        checkoutservice-2            83  2022-03-21 07:32:00 +0800
50  recommendationservice-2            80  2022-03-21 07:30:00 +0800
36                   node-6            77  2022-03-21 07:30:00 +0800
12        checkoutservice-0            76  2022-03-21 07:30:00 +0800
9             cartservice-2            73  2022-03-21 07:30:00 +0800
0                 adservice            73  2022-03-21 07:30:00 +0800
35                   node-5            67  2022-03-21 07:30:00 +0800
32                   node-2            64  2022-03-21 07:30:00 +0800)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# 1) Top 10 non-node components from comp_summary_top20 (kernel variable)
# Reuse comp_summary_top20 if present
try:
    comp_summary_top20
except NameError:
    raise RuntimeError("comp_summary_top20 not found in kernel. Run previous cells first.")

# Filter out node-* and take top 10 cmdb_id
non_node_mask = ~comp_summary_top20['cmdb_id'].astype(str).str.match(r'^node-\d+$')
top_non_node = comp_summary_top20[non_node_mask].head(10).copy()
top10_non_node_list = top_non_node['cmdb_id'].tolist()

# 2) Compute global P95 and P5 per (cmdb_id, trace_name) over full trace series
# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure columns and types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

trace_thresholds_full = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
)
# Round thresholds for compactness
trace_thresholds_full['P95'] = trace_thresholds_full['P95'].round(6)
trace_thresholds_full['P5']  = trace_thresholds_full['P5'].round(6)

# 2b) Filter trace.csv to incident window (inclusive start, exclusive end) UTC+8
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# 3) From filtered traces keep rows where cmdb_id in top10_non_node OR trace_name contains any of those names
# Build regex to match any of the top10 component names in trace_name (escape names)
escaped_names = [re.escape(n) for n in top10_non_node_list]
if escaped_names:
    name_regex = "(" + "|".join(escaped_names) + ")"
    mask_trace_name = df_trace_window['trace_name'].astype(str).str.contains(name_regex, na=False)
else:
    mask_trace_name = pd.Series(False, index=df_trace_window.index)

mask_cmdb = df_trace_window['cmdb_id'].isin(top10_non_node_list)
trace_filtered_candidates = df_trace_window[mask_cmdb | mask_trace_name].copy()

# 4) For each remaining (cmdb_id, trace_name) compute counts and earliest timestamps of highs/lows using global thresholds
# Merge thresholds into filtered candidates
trace_filtered_with_thresh = trace_filtered_candidates.merge(
    trace_thresholds_full, on=['cmdb_id','trace_name'], how='left'
)

# Determine high/low per row (handle NaNs safely)
trace_filtered_with_thresh['is_high'] = trace_filtered_with_thresh['value'] > trace_filtered_with_thresh['P95']
trace_filtered_with_thresh['is_low']  = trace_filtered_with_thresh['value'] < trace_filtered_with_thresh['P5']

# Group and aggregate
def agg_trace_group(g):
    P95 = g['P95'].iloc[0]
    P5  = g['P5'].iloc[0]
    # high
    is_high = g['is_high']
    count_high = int(is_high.sum())
    earliest_high_ts = g.loc[is_high, 'ts'].min() if count_high>0 else pd.NaT
    # low
    is_low = g['is_low']
    count_low = int(is_low.sum())
    earliest_low_ts = g.loc[is_low, 'ts'].min() if count_low>0 else pd.NaT
    return pd.Series({
        'count_high': count_high,
        'earliest_high_ts': earliest_high_ts,
        'count_low': count_low,
        'earliest_low_ts': earliest_low_ts,
        'P95': P95,
        'P5': P5
    })

if not trace_filtered_with_thresh.empty:
    trace_agg = trace_filtered_with_thresh.groupby(['cmdb_id','trace_name'], as_index=False).apply(agg_trace_group).reset_index()
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count_high','earliest_high_ts','count_low','earliest_low_ts','P95','P5'])

# Format timestamps to UTC+8 strings for display
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

if not trace_agg.empty:
    trace_agg['earliest_high_ts'] = trace_agg['earliest_high_ts'].apply(fmt_ts)
    trace_agg['earliest_low_ts']  = trace_agg['earliest_low_ts'].apply(fmt_ts)
    trace_agg['P95'] = trace_agg['P95'].round(6)
    trace_agg['P5']  = trace_agg['P5'].round(6)

# Prepare compact output: top 20 by max(count_high,count_low) descending
if not trace_agg.empty:
    trace_agg['max_count'] = trace_agg[['count_high','count_low']].max(axis=1)
    trace_top20 = trace_agg.sort_values('max_count', ascending=False).head(20)[
        ['cmdb_id','trace_name','count_high','earliest_high_ts','count_low','earliest_low_ts','P95','P5']
    ].reset_index(drop=True)
else:
    trace_top20 = trace_agg.copy()

# Totals
total_series_scanned = int(trace_agg.shape[0])
series_with_anomaly = int(trace_agg[(trace_agg['count_high']>0) | (trace_agg['count_low']>0)].shape[0])

# Keep full intermediates in kernel variables: trace_thresholds_full, trace_filtered_candidates (already set)
trace_thresholds_full, trace_filtered_candidates, trace_top20, total_series_scanned, series_with_anomaly

```
Out[4]:
```
```python
# Summarize the requested results in plain English using existing kernel variables.
try:
    top10 = top10_non_node_list
except NameError:
    try:
        top10 = comp_summary_top20[~comp_summary_top20['cmdb_id'].astype(str).str.match(r'^node-\d+$')].head(10)['cmdb_id'].tolist()
    except Exception:
        top10 = []

# Gather counts from kernel variables (fallback-safe)
n_trace_thresholds = int(trace_thresholds_full.shape[0]) if 'trace_thresholds_full' in globals() else None
n_trace_window_rows = int(df_trace_window.shape[0]) if 'df_trace_window' in globals() else None
n_filtered_candidates = int(trace_filtered_candidates.shape[0]) if 'trace_filtered_candidates' in globals() else None
n_series_scanned = int(total_series_scanned) if 'total_series_scanned' in globals() else None
n_series_with_anomaly = int(series_with_anomaly) if 'series_with_anomaly' in globals() else None

# Prepare a few top examples from trace_top20 (if present)
examples = []
if 'trace_top20' in globals() and not trace_top20.empty:
    for _, r in trace_top20.head(5).iterrows():
        examples.append(
            f"{r.cmdb_id} | {r.trace_name} -> count_high={int(r.count_high)}, earliest_high={r.earliest_high_ts or ''}; "
            f"count_low={int(r.count_low)}, earliest_low={r.earliest_low_ts or ''}; P95={r.P95}, P5={r.P5}"
        )

summary = (
    "Trace analysis summary (UTC+8):\n\n"
    "1) Top-10 non-node components selected (service/pod names):\n"
    f"- {top10}\n\n"
    "2) Trace thresholds and window filtering:\n"
    f"- Global per-series trace thresholds computed: {n_trace_thresholds} (cmdb_id,trace_name) rows.\n"
    f"- Trace rows in incident window (2022-03-21 07:30:00 to 08:00:00 UTC+8): {n_trace_window_rows} rows.\n"
    f"- Filtered candidate trace rows (kept because cmdb_id in top-10 OR trace_name mentions them): {n_filtered_candidates} rows.\n\n"
    "3) Anomaly counts in the filtered trace set:\n"
    f"- Distinct (cmdb_id, trace_name) series scanned in the filtered set: {n_series_scanned}\n"
    f"- Series with at least one anomaly (value > P95 or < P5) in the window: {n_series_with_anomaly}\n\n"
    "4) Top example anomalous trace series (from the compact top-20 results):\n"
    + ("\n".join(f"- {e}" for e in examples) if examples else "- (no examples available)") +
    "\n\n"
    "Notes:\n"
    "- Full threshold table: trace_thresholds_full\n"
    "- Filtered rows kept for analysis: trace_filtered_candidates\n"
    "- Compact top-20 anomaly table: trace_top20\n"
    "- All timestamps shown and computed in Asia/Shanghai (UTC+8).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id                           trace_name        P95        P5
0           adservice-0  trace.from_frontend-0.duration_mean   0.000023  0.000016
1           adservice-0   trace.from_frontend-0.duration_p95   0.000047  0.000019
2           adservice-0     trace.from_frontend-0.error_rate   0.000000  0.000000
3           adservice-0      trace.from_frontend-0.row_count  13.550000  6.450000
4           adservice-0  trace.from_frontend-1.duration_mean   0.000023  0.000017
...                 ...                                  ...        ...       ...
1075  shippingservice-2      trace.from_frontend-1.row_count   6.650000  1.000000
1076  shippingservice-2  trace.from_frontend-2.duration_mean   0.000070  0.000057
1077  shippingservice-2   trace.from_frontend-2.duration_p95   0.000078  0.000057
1078  shippingservice-2     trace.from_frontend-2.error_rate   0.000000  0.000000
1079  shippingservice-2      trace.from_frontend-2.row_count   3.000000  1.000000

[1080 rows x 4 columns],         timestamp            cmdb_id                           trace_name      value                        ts
0      1647819000        adservice-0  trace.from_frontend-0.duration_mean   0.000017 2022-03-21 07:30:00+08:00
1      1647819000        adservice-0   trace.from_frontend-0.duration_p95   0.000022 2022-03-21 07:30:00+08:00
2      1647819000        adservice-0     trace.from_frontend-0.error_rate   0.000000 2022-03-21 07:30:00+08:00
3      1647819000        adservice-0      trace.from_frontend-0.row_count  13.000000 2022-03-21 07:30:00+08:00
4      1647819000        adservice-0  trace.from_frontend-1.duration_mean   0.000024 2022-03-21 07:30:00+08:00
...           ...                ...                                  ...        ...                       ...
27907  1647820740  shippingservice-2      trace.from_frontend-0.row_count   5.000000 2022-03-21 07:59:00+08:00
27908  1647820740  shippingservice-2  trace.from_frontend-1.duration_mean   0.000062 2022-03-21 07:59:00+08:00
27909  1647820740  shippingservice-2   trace.from_frontend-1.duration_p95   0.000066 2022-03-21 07:59:00+08:00
27910  1647820740  shippingservice-2     trace.from_frontend-1.error_rate   0.000000 2022-03-21 07:59:00+08:00
27911  1647820740  shippingservice-2      trace.from_frontend-1.row_count   3.000000 2022-03-21 07:59:00+08:00

[22304 rows x 5 columns],                     cmdb_id                                      trace_name  count_high           earliest_high_ts  count_low            earliest_low_ts        P95        P5
0               adservice-0              trace.from_frontend-1.duration_p95           2  2022-03-21 07:30:00 +0800          1  2022-03-21 07:56:00 +0800   0.000048  0.000019
1         shippingservice-0             trace.from_frontend-1.duration_mean           2  2022-03-21 07:41:00 +0800          1  2022-03-21 07:40:00 +0800   0.000096  0.000067
2         shippingservice-0              trace.from_frontend-1.duration_p95           2  2022-03-21 07:41:00 +0800          1  2022-03-21 07:40:00 +0800   0.000160  0.000067
3         shippingservice-0                 trace.from_frontend-1.row_count           2  2022-03-21 07:45:00 +0800          0                              6.700000  1.000000
4         shippingservice-0             trace.from_frontend-2.duration_mean           2  2022-03-21 07:31:00 +0800          0                              0.000081  0.000066
5         shippingservice-0              trace.from_frontend-2.duration_p95           2  2022-03-21 07:31:00 +0800          0                              0.000104  0.000066
6               adservice-0             trace.from_frontend-1.duration_mean           2  2022-03-21 07:30:00 +0800          1  2022-03-21 07:56:00 +0800   0.000023  0.000017
7               adservice-0                 trace.from_frontend-0.row_count           2  2022-03-21 07:54:00 +0800          2  2022-03-21 07:55:00 +0800  13.550000  6.450000
8               adservice-0              trace.from_frontend-0.duration_p95           2  2022-03-21 07:47:00 +0800          0                              0.000047  0.000019
9         shippingservice-0             trace.from_frontend-0.duration_mean           2  2022-03-21 07:33:00 +0800          1  2022-03-21 07:56:00 +0800   0.000092  0.000066
10        shippingservice-0              trace.from_frontend-0.duration_p95           2  2022-03-21 07:33:00 +0800          1  2022-03-21 07:56:00 +0800   0.000148  0.000066
11        shippingservice-0                 trace.from_frontend-0.row_count           2  2022-03-21 07:53:00 +0800          0                              6.200000  1.000000
12  recommendationservice-2  trace.to_productcatalogservice-1.duration_mean           2  2022-03-21 07:50:00 +0800          1  2022-03-21 07:58:00 +0800   0.000032  0.000027
13  recommendationservice-2   trace.to_productcatalogservice-1.duration_p95           2  2022-03-21 07:50:00 +0800          0                              0.000045  0.000029
14  recommendationservice-2      trace.to_productcatalogservice-1.row_count           2  2022-03-21 07:31:00 +0800          2  2022-03-21 07:57:00 +0800  11.550000  8.450000
15              adservice-1                 trace.from_frontend-0.row_count           2  2022-03-21 07:54:00 +0800          0                             13.550000  6.000000
16              adservice-1              trace.from_frontend-0.duration_p95           2  2022-03-21 07:34:00 +0800          1  2022-03-21 07:53:00 +0800   0.000050  0.000017
17               frontend-2               trace.to_adservice-1.duration_p95           2  2022-03-21 07:51:00 +0800          0                              0.000062  0.000016
18               frontend-2                  trace.to_adservice-1.row_count           0                                     2  2022-03-21 07:39:00 +0800   7.000000  2.450000
19               frontend-2                  trace.to_adservice-2.row_count           1  2022-03-21 07:42:00 +0800          2  2022-03-21 07:41:00 +0800   7.000000  2.450000, 832, 536)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_faults_full from kernel
try:
    metric_faults_full
except NameError:
    raise RuntimeError("metric_faults_full not found in kernel. Run previous steps first.")

# Ensure start_ts is timezone-aware datetime
if not pd.api.types.is_datetime64tz_dtype(metric_faults_full['start_ts']):
    # try to convert/attach tz
    metric_faults_full['start_ts'] = pd.to_datetime(metric_faults_full['start_ts'], utc=True).dt.tz_convert(tz)

# Helper to format timestamps to UTC+8 strings
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

# Split services/pods vs nodes
is_node = metric_faults_full['cmdb_id'].astype(str).str.match(r'^node-\d+$')
df_services = metric_faults_full[~is_node].copy()
df_nodes    = metric_faults_full[is_node].copy()

# Function to build summary for a dataframe (services or nodes)
def build_comp_summary(df):
    if df.empty:
        return pd.DataFrame(columns=['cmdb_id','total_faults','earliest_fault_start',
                                     'top_fault_kpi','top_kpi_faults_count','top_kpi_earliest_start'])
    # total faults and earliest start per component
    comp_summary = df.groupby('cmdb_id', as_index=False).agg(
        total_faults=('cmdb_id','count'),
        earliest_fault_start_ts=('start_ts','min')
    )
    # per (cmdb_id,kpi_name) counts and earliest start
    comp_kpi = df.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        kpi_faults_count=('kpi_name','count'),
        kpi_earliest_start_ts=('start_ts','min')
    )
    # For each component pick top kpi by kpi_faults_count (break ties by earliest start)
    comp_kpi_sorted = comp_kpi.sort_values(['cmdb_id','kpi_faults_count','kpi_earliest_start_ts'],
                                          ascending=[True, False, True])
    top_kpi_per_comp = comp_kpi_sorted.groupby('cmdb_id', as_index=False).first()[
        ['cmdb_id','kpi_name','kpi_faults_count','kpi_earliest_start_ts']
    ].rename(columns={
        'kpi_name':'top_fault_kpi',
        'kpi_faults_count':'top_kpi_faults_count',
        'kpi_earliest_start_ts':'top_kpi_earliest_start_ts'
    })
    # Merge
    comp_full = comp_summary.merge(top_kpi_per_comp, on='cmdb_id', how='left')
    # Format timestamps for display
    comp_full['earliest_fault_start'] = comp_full['earliest_fault_start_ts'].apply(fmt_ts)
    comp_full['top_kpi_earliest_start'] = comp_full['top_kpi_earliest_start_ts'].apply(fmt_ts)
    # Select and order columns
    comp_display = comp_full[
        ['cmdb_id','total_faults','earliest_fault_start','top_fault_kpi','top_kpi_faults_count','top_kpi_earliest_start']
    ].copy()
    return comp_full, comp_display

# Build summaries and keep full tables in kernel variables
comp_fault_summary_services_full, comp_fault_summary_services = build_comp_summary(df_services)
comp_fault_summary_nodes_full, comp_fault_summary_nodes = build_comp_summary(df_nodes)

# Sort displays by earliest_fault_start ascending and limit to 20 rows
comp_fault_summary_services = comp_fault_summary_services.sort_values('earliest_fault_start').head(20).reset_index(drop=True)
comp_fault_summary_nodes    = comp_fault_summary_nodes.sort_values('earliest_fault_start').head(20).reset_index(drop=True)

# Counts of distinct components in each table
num_services_components = int(comp_fault_summary_services_full.shape[0])
num_node_components     = int(comp_fault_summary_nodes_full.shape[0])

# Return compact tables and counts (as per rules, use variable names for display)
comp_fault_summary_services, comp_fault_summary_nodes, num_services_components, num_node_components

```
Out[5]:
```
```python
# Summarize the two compact summary tables from metric_faults_full (timestamps in UTC+8).
# Uses kernel variables comp_fault_summary_services and comp_fault_summary_nodes.

try:
    svc_tbl = comp_fault_summary_services
    node_tbl = comp_fault_summary_nodes
except NameError:
    raise RuntimeError("Expected kernel variables comp_fault_summary_services and comp_fault_summary_nodes not found.")

# Totals (from previously computed full summary counts)
try:
    total_services = num_services_components
    total_nodes = num_node_components
except NameError:
    total_services = svc_tbl.shape[0]
    total_nodes = node_tbl.shape[0]

# Prepare a few example rows for each table (up to 6)
def fmt_examples(df, n=6):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append(
            f"{r.cmdb_id}: total_faults={int(r.total_faults)}, earliest_start={r.earliest_fault_start}, "
            f"top_fault_kpi={r.top_fault_kpi} ({int(r.top_kpi_faults_count)} faults, earliest {r.top_kpi_earliest_start})"
        )
    return rows

svc_examples = fmt_examples(svc_tbl, 6)
node_examples = fmt_examples(node_tbl, 6)

summary = (
    "Metric faults — compact summaries (timestamps in UTC+8):\n\n"
    f"1) Service/Pod summary (excluding node-*):\n"
    f"- Distinct service/pod components: {total_services}\n"
    "- Top example components (up to 6 shown):\n"
    + "\n".join(f"  - {line}" for line in svc_examples)
    + "\n\n"
    "Notes for services/pods:\n"
    "- Many components show earliest_fault_start = 2022-03-21 07:30:00 +0800 (incident window start).\n"
    "- Examples include frontend-2 (109 faults, top KPI with 5 faults), checkoutservice-1 (140 faults, top KPI with 6 faults),\n"
    "  currencyservice-0 (110 faults), productcatalogservice-1 (109 faults).\n\n"
    "2) Node summary (node-*):\n"
    f"- Distinct node components: {total_nodes}\n"
    "- Node examples (up to 6 shown):\n"
    + "\n".join(f"  - {line}" for line in node_examples)
    + "\n\n"
    "Notes for nodes:\n"
    "- Nodes with faults include node-1..node-6. Example: node-1 had 86 faults (earliest 2022-03-21 07:30:00 +0800) with top KPI system.cpu.iowait.\n"
    "- These node-level faults suggest system-level resource/IO signals during the incident window.\n\n"
    "Data locations in kernel:\n"
    "- Full per-component service summary: comp_fault_summary_services_full\n"
    "- Displayed compact service summary: comp_fault_summary_services\n"
    "- Full per-node summary: comp_fault_summary_nodes_full\n"
    "- Displayed compact node summary: comp_fault_summary_nodes\n\n"
    "Use the kernel tables above for detailed inspection of specific KPIs and timestamps."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  total_faults       earliest_fault_start                                      top_fault_kpi  top_kpi_faults_count     top_kpi_earliest_start
0                  adservice            73  2022-03-21 07:30:00 +0800  runtime.java_lang_GarbageCollector_LastGcInfo_...                     4  2022-03-21 07:32:00 +0800
1                 frontend-2           109  2022-03-21 07:30:00 +0800  mesh.destination.unknown.frontend.istio_reques...                     5  2022-03-21 07:34:00 +0800
2                frontend2-0            25  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_pa...                     5  2022-03-21 07:35:00 +0800
3             emailservice-1            27  2022-03-21 07:30:00 +0800  container.node-6.container_network_receive_pac...                     3  2022-03-21 07:30:00 +0800
4           paymentservice-2            37  2022-03-21 07:30:00 +0800  container.node-6.container_network_receive_MB....                     4  2022-03-21 07:34:00 +0800
5         currencyservice2-0            20  2022-03-21 07:30:00 +0800            container.node-6.container_memory_cache                     5  2022-03-21 07:32:00 +0800
6          paymentservice2-0            13  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_MB...                     3  2022-03-21 07:30:00 +0800
7      productcatalogservice             5  2022-03-21 07:30:00 +0800                                     app.grpc.count                     3  2022-03-21 07:30:00 +0800
8          currencyservice-0           110  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_MB...                     5  2022-03-21 07:33:00 +0800
9    productcatalogservice-0           102  2022-03-21 07:30:00 +0800  container.node-6.container_network_receive_pac...                     5  2022-03-21 07:39:00 +0800
10   productcatalogservice-1           109  2022-03-21 07:30:00 +0800  mesh.destination.checkoutservice.productcatalo...                     6  2022-03-21 07:31:00 +0800
11         checkoutservice-0            76  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_MB...                     3  2022-03-21 07:32:00 +0800
12         checkoutservice-1           140  2022-03-21 07:30:00 +0800  container.node-6.container_network_receive_MB....                     6  2022-03-21 07:32:00 +0800
13             cartservice-2            73  2022-03-21 07:30:00 +0800  mesh.source.cartservice.redis-cart.istio_tcp_s...                     5  2022-03-21 07:30:00 +0800
14   recommendationservice-2            80  2022-03-21 07:30:00 +0800         container.node-6.container_cpu_cfs_periods                     5  2022-03-21 07:39:00 +0800
15  recommendationservice2-0            25  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_pa...                     5  2022-03-21 07:37:00 +0800
16               cartservice             3  2022-03-21 07:30:00 +0800                                     app.grpc.count                     2  2022-03-21 07:30:00 +0800
17              adservice2-0            39  2022-03-21 07:30:00 +0800  container.node-6.container_network_transmit_MB...                     4  2022-03-21 07:30:00 +0800
18                adservice2           100  2022-03-21 07:30:00 +0800  runtime.java_lang_GarbageCollector_LastGcInfo_...                     4  2022-03-21 07:30:00 +0800
19              redis-cart-0            17  2022-03-21 07:30:00 +0800  mesh.destination.cartservice.redis-cart.istio_...                     3  2022-03-21 07:30:00 +0800,   cmdb_id  total_faults       earliest_fault_start                 top_fault_kpi  top_kpi_faults_count     top_kpi_earliest_start
0  node-1            86  2022-03-21 07:30:00 +0800             system.cpu.iowait                     5  2022-03-21 07:30:00 +0800
1  node-2            64  2022-03-21 07:30:00 +0800  system.net.udp.out_datagrams                     6  2022-03-21 07:33:00 +0800
2  node-3            48  2022-03-21 07:30:00 +0800             system.cpu.iowait                     5  2022-03-21 07:33:00 +0800
3  node-4            60  2022-03-21 07:30:00 +0800          system.cpu.pct_usage                     5  2022-03-21 07:30:00 +0800
4  node-5            67  2022-03-21 07:30:00 +0800               system.cpu.user                     5  2022-03-21 07:30:00 +0800
5  node-6            77  2022-03-21 07:30:00 +0800               system.io.await                     5  2022-03-21 07:46:00 +0800, 53, 6)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_faults_full from kernel
try:
    metric_faults_full
except NameError:
    raise RuntimeError("metric_faults_full not found in kernel. Please run previous steps.")

# Ensure start_ts is datetime with UTC+8 tz
if not pd.api.types.is_datetime64tz_dtype(metric_faults_full['start_ts']):
    metric_faults_full['start_ts'] = pd.to_datetime(metric_faults_full['start_ts'], utc=True).dt.tz_convert(tz)

# Incident window bounds (UTC+8)
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))

# Filter faults to the incident window (start_ts within [start, end))
df_window_faults = metric_faults_full[(metric_faults_full['start_ts'] >= start) & (metric_faults_full['start_ts'] < end)].copy()

# Add start_minute floored to minute
df_window_faults['start_minute'] = df_window_faults['start_ts'].dt.floor('T')

# 1) Count faults grouped by start_minute, top 10 by count desc
faults_by_minute = df_window_faults.groupby('start_minute', as_index=False).size().rename(columns={'size':'fault_count'})
faults_top10 = faults_by_minute.sort_values('fault_count', ascending=False).head(10).copy()
# Format start_minute as string UTC+8
faults_top10['start_minute'] = faults_top10['start_minute'].dt.strftime("%Y-%m-%d %H:%M:%S %z")

# Store full table in kernel variable
faults_by_minute_full = faults_by_minute.copy()

# 2) For the top two minutes, compute top 5 components per minute
top_minutes = faults_by_minute.sort_values('fault_count', ascending=False).head(2)['start_minute'].tolist()

top_components_rows = []
top_components_full = []  # keep full intermediate
for minute in top_minutes:
    # minute is Timestamp
    grp = df_window_faults[df_window_faults['start_minute'] == minute].copy()
    if grp.empty:
        continue
    # per-component counts and earliest start
    comp_stats = grp.groupby('cmdb_id').agg(
        faults_in_minute=('cmdb_id','count'),
        earliest_start_ts=('start_ts','min')
    ).reset_index()
    # For each component, find most common kpi_name and its count among those faults
    def top_kpi_info(sub):
        vc = sub['kpi_name'].value_counts()
        top_kpi = vc.index[0]
        top_kpi_count = int(vc.iloc[0])
        return pd.Series({'top_kpi': top_kpi, 'top_kpi_count': top_kpi_count})
    kpi_info = grp.groupby('cmdb_id').apply(top_kpi_info).reset_index()
    comp_stats = comp_stats.merge(kpi_info, on='cmdb_id', how='left')
    # format earliest_start_ts
    comp_stats['earliest_start_ts'] = comp_stats['earliest_start_ts'].dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %z")
    # sort by faults_in_minute desc and take top 5
    comp_stats_sorted = comp_stats.sort_values('faults_in_minute', ascending=False).head(5).copy()
    # add minute column for context (formatted)
    minute_str = minute.strftime("%Y-%m-%d %H:%M:%S %z")
    comp_stats_sorted['start_minute'] = minute_str
    top_components_rows.append(comp_stats_sorted[['start_minute','cmdb_id','faults_in_minute','top_kpi','top_kpi_count','earliest_start_ts']])
    # store full per-minute details
    top_components_full.append((minute, comp_stats.sort_values('faults_in_minute', ascending=False).reset_index(drop=True)))

# Concatenate display table
if top_components_rows:
    top_components_per_minute = pd.concat(top_components_rows, ignore_index=True)
else:
    top_components_per_minute = pd.DataFrame(columns=['start_minute','cmdb_id','faults_in_minute','top_kpi','top_kpi_count','earliest_start_ts'])

# Keep full intermediates in kernel variables
top_components_per_minute_full = top_components_full  # list of tuples (minute, full comp_stats df)

# Final compact outputs (variables to display)
faults_top10, top_components_per_minute

```
Out[6]:
```
```python
summary = (
    "Time-clustering summary of faults (UTC+8):\n\n"
    "1) Top 10 minutes by fault count (start_minute, fault_count):\n"
    "- 2022-03-21 07:58:00 +0800: 336 faults\n"
    "- 2022-03-21 07:45:00 +0800: 230 faults\n"
    "- 2022-03-21 07:30:00 +0800: 221 faults\n"
    "- 2022-03-21 07:59:00 +0800: 206 faults\n"
    "- 2022-03-21 07:53:00 +0800: 127 faults\n"
    "- 2022-03-21 07:57:00 +0800: 123 faults\n"
    "- 2022-03-21 07:54:00 +0800: 115 faults\n"
    "- 2022-03-21 07:55:00 +0800: 113 faults\n"
    "- 2022-03-21 07:48:00 +0800: 112 faults\n"
    "- 2022-03-21 07:46:00 +0800: 109 faults\n\n"
    "2) For the top two minutes, top 5 components (per-minute) with details:\n\n"
    "Minute: 2022-03-21 07:58:00 +0800\n"
    "- frontend-1: 32 faults; top_kpi = container.node-5.container_fs_usage_MB./dev/vda1 (1 fault); earliest start = 2022-03-21 07:58:00 +0800\n"
    "- productcatalogservice-2: 23 faults; top_kpi = container.node-6.container_cpu_system_seconds (1 fault); earliest start = 2022-03-21 07:58:00 +0800\n"
    "- frontend-0: 22 faults; top_kpi = container.node-6.container_file_descriptors (1 fault); earliest start = 2022-03-21 07:58:00 +0800\n"
    "- productcatalogservice-1: 20 faults; top_kpi = container.node-6.container_cpu_system_seconds (1 fault); earliest start = 2022-03-21 07:58:00 +0800\n"
    "- recommendationservice-2: 18 faults; top_kpi = container.node-6.container_cpu_cfs_periods (1 fault); earliest start = 2022-03-21 07:58:00 +0800\n\n"
    "Minute: 2022-03-21 07:45:00 +0800\n"
    "- checkoutservice-0: 47 faults; top_kpi = container.node-6.container_cpu_usage_seconds (1 fault); earliest start = 2022-03-21 07:45:00 +0800\n"
    "- checkoutservice-2: 45 faults; top_kpi = mesh.destination.frontend.checkoutservice.istio... (1 fault); earliest start = 2022-03-21 07:45:00 +0800\n"
    "- currencyservice-1: 20 faults; top_kpi = container.node-6.container_cpu_usage_seconds (1 fault); earliest start = 2022-03-21 07:45:00 +0800\n"
    "- frontend-0: 20 faults; top_kpi = container.node-6.container_network_receive_MB... (1 fault); earliest start = 2022-03-21 07:45:00 +0800\n"
    "- currencyservice-0: 13 faults; top_kpi = container.node-6.container_memory_cache (1 fault); earliest start = 2022-03-21 07:45:00 +0800\n\n"
    "Notes:\n"
    "- Counts are fault segments whose start_ts falls in the listed minute (incident window 07:30–08:00 UTC+8).\n"
    "- Many components have multiple distinct KPIs contributing faults in the same minute; top_kpi reports the most common KPI among that component's faults in that minute (with its count).\n"
    "- All timestamps shown are Asia/Shanghai (UTC+8).\n"
    "- Full intermediate tables are available in kernel variables: faults_by_minute_full and top_components_per_minute_full."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 start_minute  fault_count
28  2022-03-21 07:58:00 +0800          336
15  2022-03-21 07:45:00 +0800          230
0   2022-03-21 07:30:00 +0800          221
29  2022-03-21 07:59:00 +0800          206
23  2022-03-21 07:53:00 +0800          127
27  2022-03-21 07:57:00 +0800          123
24  2022-03-21 07:54:00 +0800          115
25  2022-03-21 07:55:00 +0800          113
18  2022-03-21 07:48:00 +0800          112
16  2022-03-21 07:46:00 +0800          109,                 start_minute                  cmdb_id  faults_in_minute                                            top_kpi  top_kpi_count          earliest_start_ts
0  2022-03-21 07:58:00 +0800               frontend-1                32   container.node-5.container_fs_usage_MB./dev/vda1              1  2022-03-21 07:58:00 +0800
1  2022-03-21 07:58:00 +0800  productcatalogservice-2                23      container.node-6.container_cpu_system_seconds              1  2022-03-21 07:58:00 +0800
2  2022-03-21 07:58:00 +0800               frontend-0                22        container.node-6.container_file_descriptors              1  2022-03-21 07:58:00 +0800
3  2022-03-21 07:58:00 +0800  productcatalogservice-1                20      container.node-6.container_cpu_system_seconds              1  2022-03-21 07:58:00 +0800
4  2022-03-21 07:58:00 +0800  recommendationservice-2                18         container.node-6.container_cpu_cfs_periods              1  2022-03-21 07:58:00 +0800
5  2022-03-21 07:45:00 +0800        checkoutservice-0                47       container.node-6.container_cpu_usage_seconds              1  2022-03-21 07:45:00 +0800
6  2022-03-21 07:45:00 +0800        checkoutservice-2                45  mesh.destination.frontend.checkoutservice.isti...              1  2022-03-21 07:45:00 +0800
7  2022-03-21 07:45:00 +0800        currencyservice-1                20       container.node-6.container_cpu_usage_seconds              1  2022-03-21 07:45:00 +0800
8  2022-03-21 07:45:00 +0800               frontend-0                20  container.node-6.container_network_receive_MB....              1  2022-03-21 07:45:00 +0800
9  2022-03-21 07:45:00 +0800        currencyservice-0                13            container.node-6.container_memory_cache              1  2022-03-21 07:45:00 +0800)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    trace_thresholds_full
except NameError:
    # compute global thresholds if missing
    df_trace['timestamp'] = df_trace['timestamp'].astype(int)
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_thresholds_full = (
        df_trace.groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    )
    trace_thresholds_full['P95'] = trace_thresholds_full['P95'].round(6)
    trace_thresholds_full['P5']  = trace_thresholds_full['P5'].round(6)

# Ensure df_trace has ts and types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Prepare trace window for the incident window (re-use if exists)
start = tz.localize(pd.Timestamp('2022-03-21 07:30:00'))
end   = tz.localize(pd.Timestamp('2022-03-21 08:00:00'))
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# Prepare logs
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_log_window = df_log[(df_log['ts'] >= start) & (df_log['ts'] < end)].copy()

# Target minutes
target_minutes = [
    tz.localize(pd.Timestamp('2022-03-21 07:30:00')),
    tz.localize(pd.Timestamp('2022-03-21 07:58:00'))
]

# keywords for trace_name filtering
keywords = ['error_rate','row_count','duration_p95','duration_mean']
kw_regex = "(" + "|".join(map(re.escape, keywords)) + ")"

# Helper to format ts
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

# Containers for outputs (will also store full filtered tables as requested)
outputs = {}

for t in target_minutes:
    minute_key = t.strftime("%Y%m%d_%H%M%S")
    minute_label = t.strftime("%Y-%m-%d %H:%M:%S %z")
    # 2) Trace anomaly extraction for this minute
    trace_min_var = f"trace_minute_filtered_{minute_key}"
    trace_min_df = df_trace_window[df_trace_window['ts'] == t].copy()
    # filter trace_name by keywords
    trace_min_df = trace_min_df[trace_min_df['trace_name'].astype(str).str.contains(kw_regex, case=False, na=False)].copy()
    # store full filtered trace table
    globals()[trace_min_var] = trace_min_df

    if trace_min_df.empty:
        trace_anom_compact = pd.DataFrame(columns=['minute_target','cmdb_id','trace_name','value','P95','P5','anomaly_type'])
        trace_agg_compact = pd.DataFrame(columns=['cmdb_id','anomaly_count','earliest_anomaly_ts'])
    else:
        # merge thresholds
        trace_min_with_thresh = trace_min_df.merge(trace_thresholds_full, on=['cmdb_id','trace_name'], how='left')
        # determine anomaly_type, treat missing thresholds as 'normal'
        def classify_row(row):
            P95 = row['P95']
            P5 = row['P5']
            val = row['value']
            if pd.isna(P95) or pd.isna(P5):
                return 'normal'
            if val > P95:
                return 'high'
            if val < P5:
                return 'low'
            return 'normal'
        trace_min_with_thresh['anomaly_type'] = trace_min_with_thresh.apply(classify_row, axis=1)
        # compact anomalous rows
        trace_anom = trace_min_with_thresh[trace_min_with_thresh['anomaly_type'] != 'normal'].copy()
        trace_anom_compact = trace_anom[['cmdb_id','trace_name','value','P95','P5','anomaly_type']].copy()
        trace_anom_compact.insert(0, 'minute_target', minute_label)
        trace_anom_compact = trace_anom_compact.head(20).reset_index(drop=True)
        # aggregate top 10 cmdb_id by anomaly_count and earliest anomaly ts
        trace_agg = trace_anom.groupby('cmdb_id').agg(
            anomaly_count=('anomaly_type','count'),
            earliest_anomaly_ts=('ts','min')
        ).reset_index()
        trace_agg['earliest_anomaly_ts'] = trace_agg['earliest_anomaly_ts'].apply(fmt_ts)
        trace_agg_compact = trace_agg.sort_values(['anomaly_count','earliest_anomaly_ts'], ascending=[False, True]).head(10).reset_index(drop=True)

    # store outputs
    outputs[f"trace_anom_{minute_key}"] = trace_anom_compact
    outputs[f"trace_agg_{minute_key}"] = trace_agg_compact

    # 3) Log summary for this minute
    log_min_var = f"log_minute_filtered_{minute_key}"
    log_min_df = df_log_window[df_log_window['ts'] == t].copy()
    globals()[log_min_var] = log_min_df

    if log_min_df.empty:
        log_agg_compact = pd.DataFrame(columns=['cmdb_id','sum_error_count','sum_row_count'])
        log_errors_compact = pd.DataFrame(columns=['cmdb_id','sum_error_count','earliest_log_ts'])
    else:
        # pivot sums by log_name
        pivot = (log_min_df.groupby(['cmdb_id','log_name'], as_index=False)['value']
                 .sum()
                 .pivot(index='cmdb_id', columns='log_name', values='value')
                 .fillna(0))
        # ensure columns exist
        for col in ['log.error_count','log.row_count']:
            if col not in pivot.columns:
                pivot[col] = 0.0
        pivot = pivot.reset_index()
        pivot['sum_error_count'] = pivot['log.error_count'].astype(float)
        pivot['sum_row_count'] = pivot['log.row_count'].astype(float)
        log_agg_compact = pivot[['cmdb_id','sum_error_count','sum_row_count']].copy()
        log_agg_compact = log_agg_compact.sort_values(['sum_error_count','sum_row_count'], ascending=[False, False]).head(20).reset_index(drop=True)
        # list cmdb_ids with sum_error_count > 0 and earliest ts in that minute
        errors = log_agg_compact[log_agg_compact['sum_error_count']>0].copy()
        if not errors.empty:
            earliest_ts = (log_min_df.groupby('cmdb_id')['ts'].min().reset_index())
            earliest_ts['earliest_log_ts'] = earliest_ts['ts'].apply(fmt_ts)
            errors = errors.merge(earliest_ts[['cmdb_id','earliest_log_ts']], on='cmdb_id', how='left')
            log_errors_compact = errors[['cmdb_id','sum_error_count','earliest_log_ts']].reset_index(drop=True)
        else:
            log_errors_compact = pd.DataFrame(columns=['cmdb_id','sum_error_count','earliest_log_ts'])

    outputs[f"log_agg_{minute_key}"] = log_agg_compact
    outputs[f"log_errors_{minute_key}"] = log_errors_compact

# Prepare final compact outputs for display: for each minute show trace_anom, trace_agg, log_agg, log_errors
# Use variables named by minute keys
m1 = target_minutes[0].strftime("%Y%m%d_%H%M%S")
m2 = target_minutes[1].strftime("%Y%m%d_%H%M%S")

trace_anom_20220321_073000 = outputs.get(f"trace_anom_{m1}", pd.DataFrame())
trace_agg_20220321_073000  = outputs.get(f"trace_agg_{m1}", pd.DataFrame())
log_agg_20220321_073000    = outputs.get(f"log_agg_{m1}", pd.DataFrame())
log_errors_20220321_073000 = outputs.get(f"log_errors_{m1}", pd.DataFrame())

trace_anom_20220321_075800 = outputs.get(f"trace_anom_{m2}", pd.DataFrame())
trace_agg_20220321_075800  = outputs.get(f"trace_agg_{m2}", pd.DataFrame())
log_agg_20220321_075800    = outputs.get(f"log_agg_{m2}", pd.DataFrame())
log_errors_20220321_075800 = outputs.get(f"log_errors_{m2}", pd.DataFrame())

# Briefly show the compact results (each variable is small / limited)
trace_anom_20220321_073000, trace_agg_20220321_073000, log_agg_20220321_073000, log_errors_20220321_073000, \
trace_anom_20220321_075800, trace_agg_20220321_075800, log_agg_20220321_075800, log_errors_20220321_075800

```
Out[7]:
```
```python
# Plain-English summary of trace+log inspection for target minutes (timestamps shown in UTC+8).
summary = (
    "Focused inspection for minutes 2022-03-21 07:30:00 +0800 and 2022-03-21 07:58:00 +0800:\n\n"
    "Stored full filtered tables in kernel variables:\n"
    "- trace_minute_filtered_20220321_073000, log_minute_filtered_20220321_073000\n"
    "- trace_minute_filtered_20220321_075800, log_minute_filtered_20220321_075800\n\n"
    "Minute 2022-03-21 07:30:00 +0800 — Trace anomalies:\n"
    "- Several trace metrics were anomalous (value > P95 or < P5). Examples (anomaly_type):\n"
    "  * adservice-0: trace.from_frontend-1.duration_mean (high), duration_p95 (high)\n"
    "  * checkoutservice-0: multiple to_* duration_mean/p95 values (mix of low and high)\n"
    "  * checkoutservice-1: trace.from_root.row_count and trace.self.row_count (high)\n"
    "  * currencyservice-0/1 and frontend-0: row_count and duration metrics with highs/lows\n"
    "- Top components by anomalous trace rows (anomaly_count, earliest_anomaly_ts):\n"
    "  1) frontend-0: 9 anomalies (earliest 07:30:00)\n"
    "  2) checkoutservice-0: 5 (earliest 07:30:00)\n"
    "  3) frontend-1: 5 (earliest 07:30:00)\n"
    "  4) checkoutservice-1: 4 (earliest 07:30:00)\n"
    "  5) productcatalogservice-1: 4 (earliest 07:30:00)\n"
    "- Logs for 07:30: aggregated per-cmdb_id show many row_count values (high traffic counts) but sum_error_count = 0 for all shown components; no cmdb_id with sum_error_count > 0 in this minute.\n\n"
    "Minute 2022-03-21 07:58:00 +0800 — Trace anomalies:\n"
    "- Notable anomalous traces include:\n"
    "  * checkoutservice-2: multiple duration metrics (trace.self.duration_mean/p95 and from_frontend durations) marked HIGH\n"
    "  * several cartservice.* row_count and self.row_count marked LOW (reduced counts)\n"
    "  * adservice.* trace.from_frontend-2 metrics mostly LOW\n"
    "- Top components by anomalous trace rows (anomaly_count, earliest_anomaly_ts):\n"
    "  1) frontend-0: 24 anomalies (earliest 07:58:00)\n"
    "  2) frontend-1: 18 (earliest 07:58:00)\n"
    "  3) frontend-2: 16 (earliest 07:58:00)\n"
    "  4) productcatalogservice-0: 15 (earliest 07:58:00)\n"
    "  5) recommendationservice-0: 12 (earliest 07:58:00)\n"
    "- Logs for 07:58: aggregated per-cmdb_id again show row_count sums but sum_error_count = 0 for all shown components; no cmdb_id with sum_error_count > 0 in this minute.\n\n"
    "Key takeaways:\n"
    "- Both minutes show many trace anomalies across frontends, checkoutservice, productcatalogservice, and other services. 07:58 has particularly many anomalies concentrated on frontends and checkoutservice-2 (high latency traces).\n"
    "- Log data in these minutes show high row_count volumes but no log.error_count > 0 for the displayed components (no direct log error spikes detected in these minute-level aggregates).\n"
    "- For deeper investigation, inspect the stored full filtered tables listed above and the compact anomaly tables created by the previous steps (trace anomaly rows and per-component anomaly aggregates for each minute).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                minute_target            cmdb_id                                      trace_name      value        P95         P5 anomaly_type
0   2022-03-21 07:30:00 +0800        adservice-0             trace.from_frontend-1.duration_mean   0.000024   0.000023   0.000017         high
1   2022-03-21 07:30:00 +0800        adservice-0              trace.from_frontend-1.duration_p95   0.000057   0.000048   0.000019         high
2   2022-03-21 07:30:00 +0800  checkoutservice-0        trace.to_currencyservice-2.duration_mean   0.000116   0.000157   0.000121          low
3   2022-03-21 07:30:00 +0800  checkoutservice-0         trace.to_paymentservice-0.duration_mean   0.000159   0.000157   0.000139         high
4   2022-03-21 07:30:00 +0800  checkoutservice-0  trace.to_productcatalogservice-1.duration_mean   0.004509   0.005994   0.004510          low
5   2022-03-21 07:30:00 +0800  checkoutservice-0   trace.to_productcatalogservice-1.duration_p95   0.004509   0.006194   0.004510          low
6   2022-03-21 07:30:00 +0800  checkoutservice-0        trace.to_shippingservice-2.duration_mean   0.000057   0.000071   0.000058          low
7   2022-03-21 07:30:00 +0800  checkoutservice-1                       trace.from_root.row_count   9.000000   8.750000   3.000000         high
8   2022-03-21 07:30:00 +0800  checkoutservice-1                            trace.self.row_count  24.000000  23.750000   8.000000         high
9   2022-03-21 07:30:00 +0800  checkoutservice-1      trace.to_productcatalogservice-0.row_count   3.000000   2.000000   1.000000         high
10  2022-03-21 07:30:00 +0800  checkoutservice-1            trace.to_shippingservice-0.row_count   3.000000   2.000000   1.000000         high
11  2022-03-21 07:30:00 +0800  currencyservice-0                 trace.from_frontend-0.row_count  64.000000  62.200000  38.450000         high
12  2022-03-21 07:30:00 +0800  currencyservice-1             trace.from_frontend-0.duration_mean   0.000111   0.000143   0.000116          low
13  2022-03-21 07:30:00 +0800  currencyservice-1              trace.from_frontend-0.duration_p95   0.000136   0.000160   0.000139          low
14  2022-03-21 07:30:00 +0800  currencyservice-1                 trace.from_frontend-0.row_count  63.000000  61.650000  37.900000         high
15  2022-03-21 07:30:00 +0800  currencyservice-2      trace.from_checkoutservice-0.duration_mean   0.000116   0.000157   0.000121          low
16  2022-03-21 07:30:00 +0800  currencyservice-2                 trace.from_frontend-0.row_count  63.000000  61.650000  38.450000         high
17  2022-03-21 07:30:00 +0800         frontend-0                        trace.self.duration_mean   0.004594   0.103481   0.004613          low
18  2022-03-21 07:30:00 +0800         frontend-0            trace.to_currencyservice-0.row_count  64.000000  62.200000  38.450000         high
19  2022-03-21 07:30:00 +0800         frontend-0        trace.to_currencyservice-1.duration_mean   0.000111   0.000143   0.000116          low,                    cmdb_id  anomaly_count        earliest_anomaly_ts
0               frontend-0              9  2022-03-21 07:30:00 +0800
1        checkoutservice-0              5  2022-03-21 07:30:00 +0800
2               frontend-1              5  2022-03-21 07:30:00 +0800
3        checkoutservice-1              4  2022-03-21 07:30:00 +0800
4  productcatalogservice-1              4  2022-03-21 07:30:00 +0800
5        currencyservice-1              3  2022-03-21 07:30:00 +0800
6              adservice-0              2  2022-03-21 07:30:00 +0800
7        currencyservice-2              2  2022-03-21 07:30:00 +0800
8  productcatalogservice-2              2  2022-03-21 07:30:00 +0800
9        currencyservice-0              1  2022-03-21 07:30:00 +0800, log_name                  cmdb_id  sum_error_count  sum_row_count
0                      frontend-0              0.0          820.0
1                      frontend-1              0.0          777.0
2         productcatalogservice-0              0.0          458.0
3         productcatalogservice-2              0.0          448.0
4         productcatalogservice-1              0.0          447.0
5                      frontend-2              0.0          443.0
6               currencyservice-1              0.0          441.0
7               currencyservice-0              0.0          439.0
8               currencyservice-2              0.0          428.0
9                   cartservice-2              0.0          413.0
10                  cartservice-1              0.0          412.0
11                  cartservice-0              0.0          394.0
12        recommendationservice-0              0.0           99.0
13        recommendationservice-2              0.0           98.0
14        recommendationservice-1              0.0           97.0
15                    adservice-0              0.0           68.0
16                    adservice-2              0.0           56.0
17              checkoutservice-1              0.0           52.0
18                    adservice-1              0.0           50.0
19              checkoutservice-0              0.0           48.0, Empty DataFrame
Columns: [cmdb_id, sum_error_count, earliest_log_ts]
Index: [],                 minute_target            cmdb_id                                      trace_name      value        P95         P5 anomaly_type
0   2022-03-21 07:58:00 +0800        adservice-0             trace.from_frontend-2.duration_mean   0.000015   0.000022   0.000016          low
1   2022-03-21 07:58:00 +0800        adservice-0              trace.from_frontend-2.duration_p95   0.000016   0.000031   0.000017          low
2   2022-03-21 07:58:00 +0800        adservice-0                 trace.from_frontend-2.row_count   2.000000   7.000000   3.000000          low
3   2022-03-21 07:58:00 +0800        adservice-1                 trace.from_frontend-2.row_count   2.000000   7.000000   2.450000          low
4   2022-03-21 07:58:00 +0800        adservice-2                 trace.from_frontend-2.row_count   2.000000   7.000000   2.450000          low
5   2022-03-21 07:58:00 +0800      cartservice-0                 trace.from_frontend-0.row_count  12.000000  17.550000  13.000000          low
6   2022-03-21 07:58:00 +0800      cartservice-1                 trace.from_frontend-0.row_count  12.000000  17.000000  13.000000          low
7   2022-03-21 07:58:00 +0800      cartservice-1                            trace.self.row_count  34.000000  49.000000  36.450000          low
8   2022-03-21 07:58:00 +0800      cartservice-2                            trace.self.row_count  34.000000  52.000000  36.800000          low
9   2022-03-21 07:58:00 +0800  checkoutservice-0        trace.to_currencyservice-1.duration_mean   0.000112   0.000152   0.000121          low
10  2022-03-21 07:58:00 +0800  checkoutservice-0         trace.to_currencyservice-1.duration_p95   0.000112   0.000159   0.000121          low
11  2022-03-21 07:58:00 +0800  checkoutservice-2             trace.from_frontend-1.duration_mean   0.034681   0.432602   0.035582          low
12  2022-03-21 07:58:00 +0800  checkoutservice-2              trace.from_frontend-1.duration_p95   0.034681   0.432602   0.035582          low
13  2022-03-21 07:58:00 +0800  checkoutservice-2             trace.from_frontend-2.duration_mean   0.610405   0.366492   0.045656         high
14  2022-03-21 07:58:00 +0800  checkoutservice-2              trace.from_frontend-2.duration_p95   0.610405   0.366492   0.045656         high
15  2022-03-21 07:58:00 +0800  checkoutservice-2                        trace.self.duration_mean   0.060882   0.058014   0.004184         high
16  2022-03-21 07:58:00 +0800  checkoutservice-2                         trace.self.duration_p95   0.316721   0.289164   0.007498         high
17  2022-03-21 07:58:00 +0800  checkoutservice-2  trace.to_productcatalogservice-1.duration_mean   0.004471   0.006898   0.004657          low
18  2022-03-21 07:58:00 +0800  checkoutservice-2   trace.to_productcatalogservice-1.duration_p95   0.004471   0.008194   0.004658          low
19  2022-03-21 07:58:00 +0800  currencyservice-0                 trace.from_frontend-0.row_count  38.000000  62.200000  38.450000          low,                    cmdb_id  anomaly_count        earliest_anomaly_ts
0               frontend-0             24  2022-03-21 07:58:00 +0800
1               frontend-1             18  2022-03-21 07:58:00 +0800
2               frontend-2             16  2022-03-21 07:58:00 +0800
3  productcatalogservice-0             15  2022-03-21 07:58:00 +0800
4  recommendationservice-0             12  2022-03-21 07:58:00 +0800
5  recommendationservice-1             12  2022-03-21 07:58:00 +0800
6  recommendationservice-2             11  2022-03-21 07:58:00 +0800
7        checkoutservice-2              8  2022-03-21 07:58:00 +0800
8  productcatalogservice-1              6  2022-03-21 07:58:00 +0800
9        currencyservice-1              4  2022-03-21 07:58:00 +0800, log_name                  cmdb_id  sum_error_count  sum_row_count
0                      frontend-0              0.0          638.0
1                      frontend-1              0.0          618.0
2         productcatalogservice-2              0.0          381.0
3         productcatalogservice-0              0.0          380.0
4         productcatalogservice-1              0.0          378.0
5                   cartservice-2              0.0          331.0
6                      frontend-2              0.0          331.0
7                   cartservice-0              0.0          321.0
8                   cartservice-1              0.0          311.0
9               currencyservice-0              0.0          271.0
10              currencyservice-2              0.0          261.0
11              currencyservice-1              0.0          259.0
12        recommendationservice-0              0.0           86.0
13        recommendationservice-1              0.0           86.0
14        recommendationservice-2              0.0           85.0
15                    adservice-0              0.0           56.0
16                    adservice-2              0.0           46.0
17                    adservice-1              0.0           42.0
18              shippingservice-0              0.0           32.0
19              shippingservice-1              0.0           27.0, Empty DataFrame
Columns: [cmdb_id, sum_error_count, earliest_log_ts]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_faults_full and thresholds if present
try:
    metric_faults_full
except NameError:
    raise RuntimeError("metric_faults_full not found in kernel. Run previous steps first.")

# Ensure start_ts is timezone-aware datetime in UTC+8
if not pd.api.types.is_datetime64tz_dtype(metric_faults_full['start_ts']):
    metric_faults_full['start_ts'] = pd.to_datetime(metric_faults_full['start_ts'], utc=True).dt.tz_convert(tz)

# Helper to format timestamps
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

# Target minutes
targets = [
    tz.localize(pd.Timestamp('2022-03-21 07:30:00')),
    tz.localize(pd.Timestamp('2022-03-21 07:58:00'))
]

# Regex for node detection
node_re = re.compile(r'^node-\d+$')

# Prepare trace thresholds if available
try:
    trace_thresholds_full
except NameError:
    # safe fallback: empty df
    trace_thresholds_full = pd.DataFrame(columns=['cmdb_id','trace_name','P95','P5'])

results = {}

for t in targets:
    key = t.strftime("%Y%m%d_%H%M%S")
    minute_label = t.strftime("%Y-%m-%d %H:%M:%S %z")
    # 1) Service/Pod metric faults for this minute (exclude nodes)
    mask_min = metric_faults_full['start_ts'] == t
    df_min = metric_faults_full[mask_min].copy()
    df_services = df_min[~df_min['cmdb_id'].astype(str).str.match(r'^node-\d+$')].copy()
    if df_services.empty:
        svc_agg = pd.DataFrame(columns=['cmdb_id','num_metric_faults','max_metric_peak_deviation','earliest_metric_fault_ts'])
    else:
        svc_grp = df_services.groupby('cmdb_id', as_index=False).agg(
            num_metric_faults=('cmdb_id','count'),
            max_metric_peak_deviation=('peak_deviation','max'),
            earliest_metric_fault_ts=('start_ts','min')
        )
        svc_grp['max_metric_peak_deviation'] = svc_grp['max_metric_peak_deviation'].astype(float).round(6)
        svc_grp['earliest_metric_fault_ts'] = svc_grp['earliest_metric_fault_ts'].apply(fmt_ts)
        svc_agg = svc_grp.sort_values(['num_metric_faults','max_metric_peak_deviation'], ascending=[False, False]).reset_index(drop=True)
    # store full intermediate
    candidates_var = f"candidates_{key}_full"
    globals()[candidates_var] = svc_grp if not df_services.empty else svc_agg.copy()

    # 2) Trace anomalies aggregation for this minute
    trace_var = f"trace_minute_filtered_{key}"
    trace_min_df = globals().get(trace_var, pd.DataFrame()).copy()
    # Ensure ts column exists and is tz-aware
    if not trace_min_df.empty:
        if 'ts' not in trace_min_df.columns:
            trace_min_df['timestamp'] = trace_min_df['timestamp'].astype(int)
            trace_min_df['ts'] = pd.to_datetime(trace_min_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        # Merge thresholds to classify anomalies
        trace_tm = trace_min_df.merge(trace_thresholds_full, on=['cmdb_id','trace_name'], how='left')
        def classify(r):
            if pd.isna(r['P95']) or pd.isna(r['P5']):
                return 'normal'
            if r['value'] > r['P95']:
                return 'high'
            if r['value'] < r['P5']:
                return 'low'
            return 'normal'
        trace_tm['anomaly_type'] = trace_tm.apply(classify, axis=1)
        trace_anom = trace_tm[trace_tm['anomaly_type'] != 'normal'].copy()
        if trace_anom.empty:
            trace_agg = pd.DataFrame(columns=['cmdb_id','num_trace_anomalies','earliest_trace_anomaly_ts'])
        else:
            trace_agg = trace_anom.groupby('cmdb_id', as_index=False).agg(
                num_trace_anomalies=('anomaly_type','count'),
                earliest_trace_anomaly_ts=('ts','min')
            )
            trace_agg['earliest_trace_anomaly_ts'] = trace_agg['earliest_trace_anomaly_ts'].apply(fmt_ts)
    else:
        trace_agg = pd.DataFrame(columns=['cmdb_id','num_trace_anomalies','earliest_trace_anomaly_ts'])
    # store full filtered trace table var already present; also store aggregated
    trace_agg_var = f"trace_agg_{key}_full"
    globals()[trace_agg_var] = trace_agg.copy()

    # 3) Log aggregation for this minute
    log_var = f"log_minute_filtered_{key}"
    log_min_df = globals().get(log_var, pd.DataFrame()).copy()
    if log_min_df.empty:
        log_agg = pd.DataFrame(columns=['cmdb_id','sum_error_count','sum_row_count'])
    else:
        # pivot sums
        pivot = (log_min_df.groupby(['cmdb_id','log_name'], as_index=False)['value']
                 .sum()
                 .pivot(index='cmdb_id', columns='log_name', values='value')
                 .fillna(0))
        # ensure columns
        for col in ['log.error_count','log.row_count']:
            if col not in pivot.columns:
                pivot[col] = 0.0
        pivot = pivot.reset_index()
        pivot['sum_error_count'] = pivot['log.error_count'].astype(float)
        pivot['sum_row_count'] = pivot['log.row_count'].astype(float)
        log_agg = pivot[['cmdb_id','sum_error_count','sum_row_count']].copy()
    log_agg_var = f"log_agg_{key}_full"
    globals()[log_agg_var] = log_agg.copy()

    # 4) Join the three per-cmdb_id aggregates
    # Start from union of cmdb_ids present in any
    cmdbs = pd.Series(
        pd.unique(
            pd.concat([
                svc_agg['cmdb_id'] if not svc_agg.empty else pd.Series(dtype=object),
                trace_agg['cmdb_id'] if not trace_agg.empty else pd.Series(dtype=object),
                log_agg['cmdb_id'] if not log_agg.empty else pd.Series(dtype=object)
            ])
        )
    , name='cmdb_id')
    joined = pd.DataFrame({'cmdb_id': cmdbs})
    if not trace_agg.empty:
        joined = joined.merge(trace_agg, on='cmdb_id', how='left')
    else:
        joined['num_trace_anomalies'] = 0
        joined['earliest_trace_anomaly_ts'] = ""
    if not svc_agg.empty:
        joined = joined.merge(svc_agg[['cmdb_id','num_metric_faults','max_metric_peak_deviation','earliest_metric_fault_ts']], on='cmdb_id', how='left')
    else:
        joined['num_metric_faults'] = 0
        joined['max_metric_peak_deviation'] = 0.0
        joined['earliest_metric_fault_ts'] = ""
    if not log_agg.empty:
        joined = joined.merge(log_agg, on='cmdb_id', how='left')
    else:
        joined['sum_error_count'] = 0.0
        joined['sum_row_count'] = 0.0

    # Fill NaNs
    joined['num_trace_anomalies'] = joined['num_trace_anomalies'].fillna(0).astype(int)
    joined['num_metric_faults'] = joined['num_metric_faults'].fillna(0).astype(int)
    joined['max_metric_peak_deviation'] = joined['max_metric_peak_deviation'].fillna(0).astype(float).round(6)
    joined['sum_error_count'] = joined['sum_error_count'].fillna(0).astype(float)
    joined['sum_row_count'] = joined['sum_row_count'].fillna(0).astype(float)
    # Replace NaT/NaN earliest timestamps with ""
    joined['earliest_trace_anomaly_ts'] = joined.get('earliest_trace_anomaly_ts', pd.Series([""]*len(joined)))
    joined['earliest_metric_fault_ts'] = joined.get('earliest_metric_fault_ts', pd.Series([""]*len(joined)))

    # Add minute_target column
    joined.insert(0, 'minute_target', minute_label)

    # Sort as requested: num_trace_anomalies desc, max_metric_peak_deviation desc, num_metric_faults desc
    joined_sorted = joined.sort_values(['num_trace_anomalies','max_metric_peak_deviation','num_metric_faults'], ascending=[False, False, False]).reset_index(drop=True)

    # Keep top 10 for display
    joined_top10 = joined_sorted.head(10).copy()

    # store full candidates
    globals()[f"candidates_{key}_full"] = joined_sorted.copy()
    results[f"svc_candidates_{key}"] = joined_top10

    # 5) Node-level check for this minute
    df_nodes_min = df_min[df_min['cmdb_id'].astype(str).str.match(r'^node-\d+$')].copy()
    if df_nodes_min.empty:
        nodes_agg = pd.DataFrame(columns=['minute_target','cmdb_id','num_metric_faults','max_metric_peak_deviation','earliest_metric_fault_ts'])
    else:
        nodes_grp = df_nodes_min.groupby('cmdb_id', as_index=False).agg(
            num_metric_faults=('cmdb_id','count'),
            max_metric_peak_deviation=('peak_deviation','max'),
            earliest_metric_fault_ts=('start_ts','min')
        )
        nodes_grp['max_metric_peak_deviation'] = nodes_grp['max_metric_peak_deviation'].astype(float).round(6)
        nodes_grp['earliest_metric_fault_ts'] = nodes_grp['earliest_metric_fault_ts'].apply(fmt_ts)
        nodes_agg = nodes_grp.sort_values('max_metric_peak_deviation', ascending=False).reset_index(drop=True)
    # Add minute_target
    if not nodes_agg.empty:
        nodes_agg.insert(0, 'minute_target', minute_label)
    else:
        nodes_agg = nodes_agg

    # store full nodes candidates
    globals()[f"nodes_candidates_{key}_full"] = nodes_agg.copy()

    # keep top 5 for display
    nodes_top5 = nodes_agg.head(5).copy()

    results[f"node_candidates_{key}"] = nodes_top5

# Prepare displayed outputs: for both minutes show joined candidate tables and node checks
out1 = results[f"svc_candidates_{targets[0].strftime('%Y%m%d_%H%M%S')}"]
out1_nodes = results[f"node_candidates_{targets[0].strftime('%Y%m%d_%H%M%S')}"]
out2 = results[f"svc_candidates_{targets[1].strftime('%Y%m%d_%H%M%S')}"]
out2_nodes = results[f"node_candidates_{targets[1].strftime('%Y%m%d_%H%M%S')}"]

# Display compact results (variables)
out1, out1_nodes, out2, out2_nodes

```
Out[8]:
```
Summary of combined metric/trace/log candidates for the two target minutes (timestamps in Asia/Shanghai, UTC+8):

Minute: 2022-03-21 07:30:00 +0800
- Top service/pod candidates (ranked by trace anomalies → metric peak → metric fault count):
  1. frontend-0 — num_trace_anomalies=9, earliest_trace_anomaly=07:30:00, num_metric_faults=15, max_metric_peak_deviation≈1.490107, earliest_metric_fault=07:30:00, sum_error_count=0, sum_row_count=820
  2. checkoutservice-0 — num_trace_anomalies=5, earliest=07:30:00, num_metric_faults=1, max_peak≈0.269004, sum_error_count=0, sum_row_count=48
  3. frontend-1 — num_trace_anomalies=5, earliest=07:30:00, num_metric_faults=0 (no metric faults at that minute), sum_error_count=0, sum_row_count=777
  4. checkoutservice-1 — num_trace_anomalies=4, earliest=07:30:00, num_metric_faults=46, max_peak≈22.333333, earliest_metric_fault=07:30:00, sum_error_count=0, sum_row_count=52
  5. productcatalogservice-1 — num_trace_anomalies=4, earliest=07:30:00, num_metric_faults=4, max_peak≈0.002360, sum_row_count=447
  6. currencyservice-1 — num_trace_anomalies=3, num_metric_faults=0, sum_row_count=441
  7. adservice-0 — num_trace_anomalies=2, num_metric_faults=0, sum_row_count=68
  8. currencyservice-2 — num_trace_anomalies=2, num_metric_faults=0, sum_row_count=428
  9. productcatalogservice-2 — num_trace_anomalies=2, num_metric_faults=0, sum_row_count=448
 10. productcatalogservice-0 — num_trace_anomalies=1, num_metric_faults=5, max_peak≈0.2, sum_row_count=458

- Node-level check (top 5 by max_metric_peak_deviation):
  - node-2 — num_metric_faults=7, max_peak≈11.333333, earliest=07:30:00
  - node-1 — num_metric_faults=3, max_peak≈6.109589, earliest=07:30:00
  - node-4 — num_metric_faults=2, max_peak≈0.467780, earliest=07:30:00
  - node-5 — num_metric_faults=3, max_peak≈0.381773, earliest=07:30:00
  - node-3 — num_metric_faults=2, max_peak≈0.081218, earliest=07:30:00

Minute: 2022-03-21 07:58:00 +0800
- Top service/pod candidates:
  1. frontend-0 — num_trace_anomalies=24, earliest_trace_anomaly=07:58:00, num_metric_faults=22, max_peak≈7.209237, earliest_metric_fault=07:58:00, sum_error_count=0, sum_row_count=638
  2. frontend-1 — num_trace_anomalies=18, num_metric_faults=32, max_peak≈0.171717, sum_row_count=618
  3. frontend-2 — num_trace_anomalies=16, num_metric_faults=9, max_peak≈0.534884, sum_row_count=331
  4. productcatalogservice-0 — num_trace_anomalies=15, num_metric_faults=17, max_peak≈0.147106, sum_row_count=380
  5. recommendationservice-1 — num_trace_anomalies=12, num_metric_faults=16, max_peak≈0.189769, sum_row_count=86
  6. recommendationservice-0 — num_trace_anomalies=12, num_metric_faults=13, max_peak≈0.113722, sum_row_count=86
  7. recommendationservice-2 — num_trace_anomalies=11, num_metric_faults=18, max_peak≈0.161696, sum_row_count=85
  8. checkoutservice-2 — num_trace_anomalies=8, num_metric_faults=2, max_peak≈0.000084, sum_row_count=15
  9. productcatalogservice-1 — num_trace_anomalies=6, num_metric_faults=20, max_peak≈0.161290, sum_row_count=378
 10. productcatalogservice-2 — num_trace_anomalies=4, num_metric_faults=23, max_peak≈0.25, sum_row_count=381

- Node-level check (top 5 by max_metric_peak_deviation):
  - node-6 — num_metric_faults=16, max_peak=1.000000, earliest=07:58:00
  - node-3 — num_metric_faults=7, max_peak≈0.847258, earliest=07:58:00
  - node-4 — num_metric_faults=5, max_peak≈0.173913, earliest=07:58:00
  - node-1 — num_metric_faults=2, max_peak≈0.095238, earliest=07:58:00
  - node-2 — num_metric_faults=1, max_peak≈0.074717, earliest=07:58:00

Additional observations / notes
- In both minutes sum_error_count is 0 for the displayed components (no log.error_count spikes), but log row_count is large (high traffic) for many frontends/services.
- 07:30 shows very large metric peak deviations at checkoutservice-1 and node-2 (max_peak ≫ 1), suggesting strong metric anomalies there.
- 07:58 shows heavy concentration of anomalies on frontends (frontend-0 highest) and node-6 shows many node-level metric faults.
- Recommended next steps: focus investigation on the frontends and checkoutservice nodes at the listed minutes, and inspect detailed per-KPI metrics and traces for the top candidates (kernel variables with the full tables are available for deeper inspection: candidates_20220321_073000_full, candidates_20220321_075800_full, and node candidates variables created in the kernel).


The original code execution output of IPython Kernel is also provided below for reference:

(               minute_target                  cmdb_id  num_trace_anomalies  earliest_trace_anomaly_ts  num_metric_faults  max_metric_peak_deviation   earliest_metric_fault_ts  sum_error_count  sum_row_count
0  2022-03-21 07:30:00 +0800               frontend-0                    9  2022-03-21 07:30:00 +0800                 15                   1.490107  2022-03-21 07:30:00 +0800              0.0          820.0
1  2022-03-21 07:30:00 +0800        checkoutservice-0                    5  2022-03-21 07:30:00 +0800                  1                   0.269004  2022-03-21 07:30:00 +0800              0.0           48.0
2  2022-03-21 07:30:00 +0800               frontend-1                    5  2022-03-21 07:30:00 +0800                  0                   0.000000                        NaN              0.0          777.0
3  2022-03-21 07:30:00 +0800        checkoutservice-1                    4  2022-03-21 07:30:00 +0800                 46                  22.333333  2022-03-21 07:30:00 +0800              0.0           52.0
4  2022-03-21 07:30:00 +0800  productcatalogservice-1                    4  2022-03-21 07:30:00 +0800                  4                   0.002360  2022-03-21 07:30:00 +0800              0.0          447.0
5  2022-03-21 07:30:00 +0800        currencyservice-1                    3  2022-03-21 07:30:00 +0800                  0                   0.000000                        NaN              0.0          441.0
6  2022-03-21 07:30:00 +0800              adservice-0                    2  2022-03-21 07:30:00 +0800                  0                   0.000000                        NaN              0.0           68.0
7  2022-03-21 07:30:00 +0800        currencyservice-2                    2  2022-03-21 07:30:00 +0800                  0                   0.000000                        NaN              0.0          428.0
8  2022-03-21 07:30:00 +0800  productcatalogservice-2                    2  2022-03-21 07:30:00 +0800                  0                   0.000000                        NaN              0.0          448.0
9  2022-03-21 07:30:00 +0800  productcatalogservice-0                    1  2022-03-21 07:30:00 +0800                  5                   0.200000  2022-03-21 07:30:00 +0800              0.0          458.0,                minute_target cmdb_id  num_metric_faults  max_metric_peak_deviation   earliest_metric_fault_ts
0  2022-03-21 07:30:00 +0800  node-2                  7                  11.333333  2022-03-21 07:30:00 +0800
1  2022-03-21 07:30:00 +0800  node-1                  3                   6.109589  2022-03-21 07:30:00 +0800
2  2022-03-21 07:30:00 +0800  node-4                  2                   0.467780  2022-03-21 07:30:00 +0800
3  2022-03-21 07:30:00 +0800  node-5                  3                   0.381773  2022-03-21 07:30:00 +0800
4  2022-03-21 07:30:00 +0800  node-3                  2                   0.081218  2022-03-21 07:30:00 +0800,                minute_target                  cmdb_id  num_trace_anomalies  earliest_trace_anomaly_ts  num_metric_faults  max_metric_peak_deviation   earliest_metric_fault_ts  sum_error_count  sum_row_count
0  2022-03-21 07:58:00 +0800               frontend-0                   24  2022-03-21 07:58:00 +0800                 22                   7.209237  2022-03-21 07:58:00 +0800              0.0          638.0
1  2022-03-21 07:58:00 +0800               frontend-1                   18  2022-03-21 07:58:00 +0800                 32                   0.171717  2022-03-21 07:58:00 +0800              0.0          618.0
2  2022-03-21 07:58:00 +0800               frontend-2                   16  2022-03-21 07:58:00 +0800                  9                   0.534884  2022-03-21 07:58:00 +0800              0.0          331.0
3  2022-03-21 07:58:00 +0800  productcatalogservice-0                   15  2022-03-21 07:58:00 +0800                 17                   0.147106  2022-03-21 07:58:00 +0800              0.0          380.0
4  2022-03-21 07:58:00 +0800  recommendationservice-1                   12  2022-03-21 07:58:00 +0800                 16                   0.189769  2022-03-21 07:58:00 +0800              0.0           86.0
5  2022-03-21 07:58:00 +0800  recommendationservice-0                   12  2022-03-21 07:58:00 +0800                 13                   0.113722  2022-03-21 07:58:00 +0800              0.0           86.0
6  2022-03-21 07:58:00 +0800  recommendationservice-2                   11  2022-03-21 07:58:00 +0800                 18                   0.161696  2022-03-21 07:58:00 +0800              0.0           85.0
7  2022-03-21 07:58:00 +0800        checkoutservice-2                    8  2022-03-21 07:58:00 +0800                  2                   0.000084  2022-03-21 07:58:00 +0800              0.0           15.0
8  2022-03-21 07:58:00 +0800  productcatalogservice-1                    6  2022-03-21 07:58:00 +0800                 20                   0.161290  2022-03-21 07:58:00 +0800              0.0          378.0
9  2022-03-21 07:58:00 +0800  productcatalogservice-2                    4  2022-03-21 07:58:00 +0800                 23                   0.250000  2022-03-21 07:58:00 +0800              0.0          381.0,                minute_target cmdb_id  num_metric_faults  max_metric_peak_deviation   earliest_metric_fault_ts
0  2022-03-21 07:58:00 +0800  node-6                 16                   1.000000  2022-03-21 07:58:00 +0800
1  2022-03-21 07:58:00 +0800  node-3                  7                   0.847258  2022-03-21 07:58:00 +0800
2  2022-03-21 07:58:00 +0800  node-4                  5                   0.173913  2022-03-21 07:58:00 +0800
3  2022-03-21 07:58:00 +0800  node-1                  2                   0.095238  2022-03-21 07:58:00 +0800
4  2022-03-21 07:58:00 +0800  node-2                  1                   0.074717  2022-03-21 07:58:00 +0800)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_faults_full from kernel
try:
    metric_faults_full
except NameError:
    raise RuntimeError("metric_faults_full not found in kernel. Run previous steps first.")

# Ensure datetime tz-aware
if not pd.api.types.is_datetime64tz_dtype(metric_faults_full['start_ts']):
    metric_faults_full['start_ts'] = pd.to_datetime(metric_faults_full['start_ts'], utc=True).dt.tz_convert(tz)
if not pd.api.types.is_datetime64tz_dtype(metric_faults_full['end_ts']):
    metric_faults_full['end_ts'] = pd.to_datetime(metric_faults_full['end_ts'], utc=True).dt.tz_convert(tz)

# Helper to format timestamps to UTC+8 string
def fmt_ts(x):
    if pd.isna(x):
        return ""
    if getattr(x, "tzinfo", None) is None:
        x = tz.localize(pd.Timestamp(x))
    else:
        x = x.tz_convert(tz)
    return x.strftime("%Y-%m-%d %H:%M:%S %z")

# Target A: 2022-03-21 07:30:00 +0800
tA = tz.localize(pd.Timestamp("2022-03-21 07:30:00"))
candidates_A = ["checkoutservice-1", "frontend-0", "frontend-1", "productcatalogservice-1", "node-2"]

maskA = (metric_faults_full['start_ts'] == tA) & (metric_faults_full['cmdb_id'].isin(candidates_A))
faults_0730_candidates_full = metric_faults_full[maskA].copy()

# Prepare display columns and formatting
if not faults_0730_candidates_full.empty:
    faults_0730_candidates_full['start_ts_str'] = faults_0730_candidates_full['start_ts'].apply(fmt_ts)
    faults_0730_candidates_full['end_ts_str'] = faults_0730_candidates_full['end_ts'].apply(fmt_ts)
    faults_0730_candidates_full['peak_deviation'] = faults_0730_candidates_full['peak_deviation'].astype(float).round(6)
    faults_0730_candidates_full['peak_value'] = faults_0730_candidates_full['peak_value'].astype(float).round(6)
    faults_0730_candidates_full['threshold_used'] = faults_0730_candidates_full['threshold_used'].astype(float).round(6)
    faults_0730_top = faults_0730_candidates_full.sort_values('peak_deviation', ascending=False).head(20)[
        ['cmdb_id','kpi_name','fault_type','start_ts_str','end_ts_str','num_points','peak_deviation','peak_value','threshold_used']
    ].rename(columns={'start_ts_str':'start_ts','end_ts_str':'end_ts'}).reset_index(drop=True)
    # summary: earliest start_ts and largest non-zero peak_deviation
    earliest_start_A = faults_0730_candidates_full['start_ts'].min()
    nonzero = faults_0730_candidates_full[faults_0730_candidates_full['peak_deviation'] > 0]
    if not nonzero.empty:
        idx = nonzero['peak_deviation'].astype(float).idxmax()
        top_row = nonzero.loc[idx]
        largest_pair_A = (top_row['cmdb_id'], top_row['kpi_name'], float(top_row['peak_deviation']))
        summary_0730 = f"earliest_start={fmt_ts(earliest_start_A)}; largest_nonzero_peak_deviation=({largest_pair_A[0]}, {largest_pair_A[1]}) = {largest_pair_A[2]:.6g}"
    else:
        summary_0730 = f"earliest_start={fmt_ts(earliest_start_A)}; no non-zero peak_deviation found among selected faults"
else:
    faults_0730_top = faults_0730_candidates_full.head(0)[['cmdb_id','kpi_name','fault_type','start_ts','end_ts','num_points','peak_deviation','peak_value','threshold_used']]
    summary_0730 = "No faults matching criteria for 2022-03-21 07:30:00 +0800"

# Expose full variable
globals()['faults_0730_candidates_full'] = faults_0730_candidates_full

# Target B: 2022-03-21 07:58:00 +0800
tB = tz.localize(pd.Timestamp("2022-03-21 07:58:00"))
candidates_B = ["frontend-0","frontend-1","frontend-2","checkoutservice-2","productcatalogservice-0","node-6"]

maskB = (metric_faults_full['start_ts'] == tB) & (metric_faults_full['cmdb_id'].isin(candidates_B))
faults_0758_candidates_full = metric_faults_full[maskB].copy()

if not faults_0758_candidates_full.empty:
    faults_0758_candidates_full['start_ts_str'] = faults_0758_candidates_full['start_ts'].apply(fmt_ts)
    faults_0758_candidates_full['end_ts_str'] = faults_0758_candidates_full['end_ts'].apply(fmt_ts)
    faults_0758_candidates_full['peak_deviation'] = faults_0758_candidates_full['peak_deviation'].astype(float).round(6)
    faults_0758_candidates_full['peak_value'] = faults_0758_candidates_full['peak_value'].astype(float).round(6)
    faults_0758_candidates_full['threshold_used'] = faults_0758_candidates_full['threshold_used'].astype(float).round(6)
    faults_0758_top = faults_0758_candidates_full.sort_values('peak_deviation', ascending=False).head(20)[
        ['cmdb_id','kpi_name','fault_type','start_ts_str','end_ts_str','num_points','peak_deviation','peak_value','threshold_used']
    ].rename(columns={'start_ts_str':'start_ts','end_ts_str':'end_ts'}).reset_index(drop=True)
    earliest_start_B = faults_0758_candidates_full['start_ts'].min()
    nonzeroB = faults_0758_candidates_full[faults_0758_candidates_full['peak_deviation'] > 0]
    if not nonzeroB.empty:
        idxb = nonzeroB['peak_deviation'].astype(float).idxmax()
        top_row_b = nonzeroB.loc[idxb]
        largest_pair_B = (top_row_b['cmdb_id'], top_row_b['kpi_name'], float(top_row_b['peak_deviation']))
        summary_0758 = f"earliest_start={fmt_ts(earliest_start_B)}; largest_nonzero_peak_deviation=({largest_pair_B[0]}, {largest_pair_B[1]}) = {largest_pair_B[2]:.6g}"
    else:
        summary_0758 = f"earliest_start={fmt_ts(earliest_start_B)}; no non-zero peak_deviation found among selected faults"
else:
    faults_0758_top = faults_0758_candidates_full.head(0)[['cmdb_id','kpi_name','fault_type','start_ts','end_ts','num_points','peak_deviation','peak_value','threshold_used']]
    summary_0758 = "No faults matching criteria for 2022-03-21 07:58:00 +0800"

# Expose full variable
globals()['faults_0758_candidates_full'] = faults_0758_candidates_full

# Final compact outputs (display these variables)
faults_0730_top, summary_0730, faults_0758_top, summary_0758

```
Out[9]:
```
```python
# Plain-English summary (UTC+8) of KPI-level inspection for the two target minutes.
summary = (
    "KPI-level findings (timestamps in Asia/Shanghai UTC+8):\n\n"
    "Minute 2022-03-21 07:30:00 +0800:\n"
    "- Earliest fault start among selected candidates: 2022-03-21 07:30:00 +0800.\n"
    "- Largest non-zero peak_deviation: checkoutservice-1 | mesh.source.checkoutservice.jaeger-collector.istio_request_duration_milliseconds.http.202. => peak_deviation ≈ 22.3333.\n"
    "- Other notable KPI-level anomalies at this minute:\n"
    "  * node-2 | system.net.tcp.retrans_segs — peak_deviation ≈ 11.3333 (multi-minute segment across 07:30–07:31).\n"
    "  * frontend-0 | mesh.source.frontend.jaeger-collector... — peak_deviation ≈ 1.4901.\n"
    "  * checkoutservice-1 has many high peaks (several mesh.source.* KPIs) with peak_deviation in the 0.2–0.4 range and very large absolute values for some KPIs.\n"
    "- Interpretation: checkoutservice-1 and node-2 show the strongest KPI-level deviations at 07:30; frontend-0 also shows a noticeable spike.\n\n"
    "Minute 2022-03-21 07:58:00 +0800:\n"
    "- Earliest fault start among selected candidates: 2022-03-21 07:58:00 +0800.\n"
    "- Largest non-zero peak_deviation: frontend-0 | mesh.source.frontend.checkoutservice.istio_request_duration_milliseconds.grpc.200.0.0 => peak_deviation ≈ 7.20924.\n"
    "- Other notable KPI-level anomalies at this minute:\n"
    "  * node-6 shows multiple node KPIs with peak_deviation = 1.0 (system.cpu.iowait, system.io.await, system.io.svctm, system.io.util) across 07:58–07:59, indicating node-level I/O/CPU symptoms.\n"
    "  * frontend-2 | container.node-5.container_cpu_cfs_periods — peak_deviation ≈ 0.5349.\n"
    "  * Many frontend-* KPIs (mesh source/destination) show moderate deviations (~0.15–0.25) with large absolute values, indicating broad frontend-related anomalies at 07:58.\n"
    "- Interpretation: 07:58 concentrates severe frontend KPI deviations (frontend-0 highest) together with node-6 resource/I/O anomalies.\n\n"
    "Where to inspect next:\n"
    "- Full KPI-level tables are stored for deeper inspection in kernel variables: faults_0730_candidates_full and faults_0758_candidates_full.\n"
    "- Prioritize checkoutservice-1 and node-2 around 07:30, and frontend-0 and node-6 around 07:58 for detailed traces/logs and metric time-series review.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name fault_type                   start_ts                     end_ts  num_points  peak_deviation    peak_value  threshold_used
0   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1       22.333333      0.024500        0.001050
1              node-2                        system.net.tcp.retrans_segs       high  2022-03-21 07:30:00 +0800  2022-03-21 07:31:00 +0800           2       11.333333      1.480000        0.120000
2          frontend-0  mesh.source.frontend.jaeger-collector.istio_re...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        1.490107      0.040900        0.016425
3              node-2                                      system.load.1       high  2022-03-21 07:30:00 +0800  2022-03-21 07:32:00 +0800           3        0.872093      3.220000        1.720000
4   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.411055    702.000000      497.500000
5   checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000     10.500000        7.500000
6   checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000  11550.000000     8250.000000
7   checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000     10.500000        7.500000
8   checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000      7.000000        5.000000
9   checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000      7.000000        5.000000
10  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000     10.500000        7.500000
11  checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000      7.000000        5.000000
12  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.400000      7.000000        5.000000
13  checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.313725   2010.000000     1530.000000
14  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.312500   2362.500000     1800.000000
15  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.312500  16275.000000    12400.000000
16  checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.273964  49175.000000    38600.000000
17  checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.254545      0.015525        0.012375
18             node-2                            system.net.tcp.out_segs       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.228455   2137.880000     1740.300000
19  checkoutservice-1  mesh.source.checkoutservice.paymentservice.ist...       high  2022-03-21 07:30:00 +0800  2022-03-21 07:30:00 +0800           1        0.227027      0.005675        0.004625, 'earliest_start=2022-03-21 07:30:00 +0800; largest_nonzero_peak_deviation=(checkoutservice-1, mesh.source.checkoutservice.jaeger-collector.istio_request_duration_milliseconds.http.202.) = 22.3333',        cmdb_id                                           kpi_name fault_type                   start_ts                     end_ts  num_points  peak_deviation   peak_value  threshold_used
0   frontend-0  mesh.source.frontend.checkoutservice.istio_req...       high  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        7.209237       2.1775        0.265250
1       node-6                                  system.cpu.iowait        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        1.000000       0.0000        0.040000
2       node-6                                    system.io.await        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        1.000000       0.0000        0.520000
3       node-6                                    system.io.svctm        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        1.000000       0.0000        0.100000
4       node-6                                     system.io.util        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        1.000000       0.0000        0.150000
5       node-6                                      system.load.1        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.680000       0.3200        1.000000
6   frontend-2         container.node-5.container_cpu_cfs_periods       high  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.534884     132.0000       86.000000
7       node-6                                      system.load.5        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.304918       2.1200        3.050000
8   frontend-2  mesh.source.frontend.currencyservice.istio_req...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        0.240315       0.0652        0.085825
9   frontend-0  mesh.source.frontend.jaeger-collector.istio_re...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.225806      48.0000       62.000000
10  frontend-0  mesh.source.frontend.jaeger-collector.istio_re...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.204918    4825.7500     6069.500000
11  frontend-2  mesh.source.frontend.currencyservice.istio_req...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        0.185185   68200.0000    83700.000000
12  frontend-1  mesh.source.frontend.checkoutservice.istio_req...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        0.171717       0.0410        0.049500
13  frontend-1  mesh.source.frontend.jaeger-collector.istio_re...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.170732      51.0000       61.500000
14  frontend-2  mesh.source.frontend.currencyservice.istio_res...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        0.169154   12525.0000    15075.000000
15  frontend-1  mesh.source.frontend.productcatalogservice.ist...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.159965   56925.0000    67765.000000
16  frontend-1  mesh.destination.unknown.frontend.istio_respon...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.159854  316525.0000   376750.000000
17  frontend-1  mesh.source.frontend.productcatalogservice.ist...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:58:00 +0800           1        0.158151  268150.0000   318525.000000
18  frontend-1  mesh.destination.unknown.frontend.istio_reques...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.155806   11432.5000    13542.500000
19  frontend-1  mesh.source.frontend.jaeger-collector.istio_re...        low  2022-03-21 07:58:00 +0800  2022-03-21 07:59:00 +0800           2        0.151637  465900.0000   549175.000000, 'earliest_start=2022-03-21 07:58:00 +0800; largest_nonzero_peak_deviation=(frontend-0, mesh.source.frontend.checkoutservice.istio_request_duration_milliseconds.grpc.200.0.0) = 7.20924')```
```